# Report performance analysis · FabricLoadTest v3

**Purpose.** Mine the CSV files written by the `FabricLoadTest_v3` load test (versions 3.1.1 to 3.6.x) with **20 analyses** designed for a
performance-testing team: overall report performance, per action/page, per visual, per user (RLS identity), aggregated times and,
above all, **the skew introduced by the staggered start (offset) and by think times**.

**Read-only.** This notebook does not start tests and does not modify any CSV. It reads, with DuckDB and without copying data into
tables, what `RunLoadTest_v3` and `RunPerfScenario_v3` write under `Files/PerfScenarios/logs/`.
The older `AnalyzeLoadTest_v3` checks that a run behaved as configured (ramp, think times, parallel models); this notebook analyses
how the **report** performed.

## How to run it
1. *Add data items* and attach, as the **default lakehouse**, the lakehouse the test writes to (the one that contains `Files/PerfScenarios/logs`).
2. Adjust the **Parameters** cell: above all `RUN_PREFIX` (= `test_name_prefix` of the pipeline; `ramp-v3` by default) and, if you want, `SINCE_UTC`, `MODELS`, `LEVELS`, `SLA_SEC`.
3. **Run all**. If `import duckdb` fails, the first code cell installs it with `%pip` (only in that case).
4. Each analysis has its header, a table and a chart. If an analysis has no data it prints `No data for ...` and the rest keeps running.
5. At the end, **Automatic conclusions** summarises the alerts; with `SAVE_TO_TABLES = True` every result is saved as a Delta table `perf_qNN_*` (so you can build a Power BI report on top) and with `EXPORT_JSON = True` as JSON files under `Files/PerfScenarios/analysis/<ANALYSIS_ID>/`, which is what the LoadTest Fabric App shows.

It works with CSV files from 3.3 (22 columns) and 3.4+ (with `status` and `error`): failed queries are **excluded** from latencies and counted separately.
Models and levels can be mixed; the cells tolerate having no think times, no RLS, an empty `visual_name`, a single level or a single model.

## Index of analyses

| No. | Analysis | What it measures | Visual |
|---|---|---|---|
| Q01 | Executive summary | KPIs per model and level: queries, percentiles, Apdex | Grouped bars p50/p90/p95 |
| Q02 | Degradation curve | How latency grows with concurrency and where the knee is | Lines per model |
| Q03 | Throughput vs latency | Queries/s and actions/min against p90 (saturation) | Bars + line |
| Q04 | Timeline | Active users, queries in flight and p90 over time | Small multiples of lines |
| Q05 | Time per action | Perceived response per page/interaction (serial and parallel bounds) | Horizontal bars |
| Q06 | Full user journey | Time spent waiting for the model across the report journey | Box plot |
| Q07 | Slowest visuals | Where to optimise first (% of total time) | Horizontal bars |
| Q08 | Visual × level heat map | Which visuals degrade under load | Heat map |
| Q09 | Visual stability | Variability (coefficient of variation) | Scatter |
| Q10 | Volume vs time | Slow because of rows returned or because of calculation? | Log-log scatter |
| Q11 | Per user / RLS performance | Atypical identities | Horizontal bars |
| Q12 | Cold vs warm cache | How much the cache helps the high levels | Grouped bars |
| Q13 | Wall-clock breakdown of a user | Offset + queries + think times + delay + overhead | 100 % stacked bars |
| Q14 | Think-time accuracy | Scheduled versus actual pause | Histogram + scatter |
| Q15 | Staggered-start skew | Scheduled offset versus actual start | Small multiples |
| Q16 | Nominal vs effective concurrency | Dilution caused by think times (duty cycle) | Bars + line |
| Q17 | Ramp phases | Ramp-up, plateau and ramp-down | Grouped bars |
| Q18 | SLA compliance per action | Interaction Apdex | 100 % stacked bars + line |
| Q19 | Completeness and errors | Expected users vs users with results, `error.log` | Grouped bars |
| Q20 | Run comparison | Before/after regression per visual | Diverging bars |

## Glossary
- **p50 / p90 / p95**: percentiles of the duration; 50/90/95 % of the queries take less than that value.
- **Apdex** (per query): (satisfied + 0.5 × tolerating) / total, where satisfied ≤ `APDEX_T` and tolerating ≤ 4 × `APDEX_T`.
- **Plateau**: the part of a run in which **all** users are active; only there is the nominal level reached.
- **Effective concurrency**: queries that are really simultaneous; with think times it is far below the nominal number of users.
- **Serial / parallel bound**: the tool runs the queries of an action one after another; `sum(duration)` is the upper bound of the perceived time and `max(duration)` the lower bound (ideal Power BI parallelism).
- **Offset**: initial wait of every user (`ramp_up_sec × thread_id / N` + jitter) to stagger the start.
- **Think time**: pause between user actions, recorded with Performance Analyzer and replayed (scale, cap and jitter).

> `duration` is client time (XMLA + row reading); it does not include browser rendering. The levels of one model run one after another **without clearing the cache** and the models run in parallel.

In [ ]:
# Installs DuckDB only if it is not available in the environment
try:
    import duckdb
except ImportError:
    get_ipython().run_line_magic("pip", "install duckdb --quiet")
    import duckdb
print("DuckDB", duckdb.__version__)

### Parameters
The next cell is tagged **Parameters**: set the test prefix, filters and thresholds here.

In [ ]:
LOGS = "/lakehouse/default/Files/PerfScenarios/logs"
RUN_PREFIX = "ramp-v3"          # test_name_prefix of the pipeline
SINCE_UTC = None                # "2026-11-20T00:00:00Z" to ignore earlier runs
RUN_IDS = None                  # list of specific loadtest_id values, or None = all
MODELS = None                   # list of model names, or None = all
LEVELS = None                   # list of levels (concurrent_threads), or None = all
FOCUS_LEVEL = None              # level used by the detailed analyses; None = the highest level of each model
SLA_SEC = 3.0                   # time objective per action (page/interaction)
APDEX_T = 1.0                   # Apdex threshold per query (satisfied <= T, tolerating <= 4T)
KNEE_FACTOR = 2.0               # p90 degradation versus the base level that marks the "knee"
BUCKET_SEC = 10                 # interval size of the time series
TOP_N = 20                      # maximum number of visuals/users in charts
TIMELINE_MAX_RUNS = 6           # maximum number of runs in the time series
BASELINE_RUN = None             # loadtest_id used as the reference in Q20 (None = automatic)
CANDIDATE_RUN = None            # loadtest_id compared in Q20 (None = automatic)
MASK_UPN = True                 # hide UPNs in tables and charts
SAVE_TO_TABLES = False          # True = save every result as a Delta table perf_qNN_*
EXPORT_JSON = False             # True = write manifest.json and qNN_<topic>.json (read by the LoadTest Fabric App) under EXPORT_ROOT/<ANALYSIS_ID>
ANALYSIS_ID = None              # folder name of the export; None = <RUN_PREFIX>-<UTC date and time>
EXPORT_ROOT = "/lakehouse/default/Files/PerfScenarios/analysis"

### Setup
DuckDB connection, views over the CSV files (`results`, `results_all`, `pauses`, `actions`, `results_focus`), the `q()` function and the chart helpers.

In [ ]:
import duckdb, os, re, glob, hashlib, math
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    display
except NameError:          # outside Fabric/Jupyter
    display = print

def _as_list(value):
    """Typed notebook parameters cannot be lists: accept a JSON list ('["A","B"]') or a comma-separated text ('A,B') as well."""
    if isinstance(value, str):
        text = value.strip()
        if not text: return None
        value = json.loads(text) if text.startswith("[") else [v.strip() for v in text.split(",") if v.strip()]
    return list(value) if value else None

import json
RUN_IDS, MODELS = _as_list(RUN_IDS), _as_list(MODELS)
LEVELS = [int(l) for l in _as_list(LEVELS)] if _as_list(LEVELS) else None
SINCE_UTC = SINCE_UTC or None
FOCUS_LEVEL = int(FOCUS_LEVEL) if FOCUS_LEVEL not in (None, "") else None
BASELINE_RUN, CANDIDATE_RUN = BASELINE_RUN or None, CANDIDATE_RUN or None

if not glob.glob(f"{LOGS}/{RUN_PREFIX}-*/*_user_*.csv"):
    raise FileNotFoundError(f"No result CSV files in {LOGS}/{RUN_PREFIX}-*/. Check RUN_PREFIX and that the default lakehouse of "
                            "this notebook is the one that holds the logs of the test.")

# Fixed dialect: the DAX text in the CSV is multi-line and quoted, and automatic detection can fail with it
CSV_OPTS = "header = true, union_by_name = true, delim = ',', quote = '\"', escape = '\"', strict_mode = false"
con = duckdb.connect()
con.execute("CREATE OR REPLACE MACRO ts(x) AS epoch_ms(CAST(x * 1000 AS BIGINT))")  # epoch s -> UTC TIMESTAMP

def _in_list(col, values):
    if not values: return ""
    vals = ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)
    return f" AND {col} IN ({vals})"

where = "WHERE 1=1"
if SINCE_UTC:
    where += f" AND start_time >= {datetime.fromisoformat(SINCE_UTC.replace('Z', '+00:00')).timestamp()}"
where += _in_list("loadtest_id", RUN_IDS) + _in_list("model", MODELS)
if LEVELS: where += f" AND concurrent_threads IN ({', '.join(str(int(l)) for l in LEVELS)})"

con.execute(f"CREATE OR REPLACE VIEW _raw AS SELECT * FROM read_csv('{LOGS}/{RUN_PREFIX}-*/*_user_*.csv', {CSV_OPTS})")
_cols = {r[0] for r in con.execute("DESCRIBE _raw").fetchall()}
HAS_STATUS = {"status", "error"} <= _cols          # CSV files from 3.4 on: failed queries are rows with status = 'error'
_excl = " EXCLUDE (status, error)" if HAS_STATUS else ""
_status = "coalesce(CAST(status AS VARCHAR), 'ok')" if HAS_STATUS else "'ok'"
_error = "coalesce(CAST(error AS VARCHAR), '')" if HAS_STATUS else "''"

# results_all: every row (failed queries too). results: successful queries only, used for latencies.
con.execute(f"""
CREATE OR REPLACE VIEW results_all AS
SELECT *{_excl},
       {_status}                                           AS status,
       {_error}                                            AS error,
       start_time + duration                               AS end_time,
       concurrent_threads                                  AS level,
       "rows"                                              AS row_count,
       coalesce(nullif(CAST(effective_username AS VARCHAR), ''), 'no_rls_thread_' || CAST(thread_id AS VARCHAR)) AS user_key,
       coalesce(nullif(CAST(visual_name AS VARCHAR), ''), '(untitled)') || ' #' || left(coalesce(CAST(visual_id AS VARCHAR), 'na'), 5) AS visual_label,
       md5(CAST(query AS VARCHAR))                         AS query_hash
FROM _raw
{where}
""")
con.execute("CREATE OR REPLACE VIEW results AS SELECT * FROM results_all WHERE status = 'ok'")

# Pauses view: if there are no think-time CSV files, create an EMPTY view with the same schema so that no query fails
try:
    con.execute(f"""
    CREATE OR REPLACE VIEW pauses AS
    SELECT *,
           epoch(CAST(sleep_start_utc AS TIMESTAMPTZ)) AS sleep_start_epoch,
           epoch(CAST(sleep_end_utc   AS TIMESTAMPTZ)) AS sleep_end_epoch
    FROM read_csv('{LOGS}/{RUN_PREFIX}-*/thinktimes/*_thinktimes.csv', {CSV_OPTS})
    WHERE loadtest_id IN (SELECT DISTINCT loadtest_id FROM results_all)
    """)
    con.execute("SELECT count(*) FROM pauses").fetchone()
    HAS_PAUSES = True
except Exception:
    HAS_PAUSES = False
    con.execute("""
    CREATE OR REPLACE VIEW pauses AS
    SELECT * FROM (VALUES (NULL::VARCHAR, NULL::VARCHAR, NULL::BIGINT, NULL::VARCHAR, NULL::BIGINT, NULL::BIGINT,
                           NULL::VARCHAR, NULL::VARCHAR, NULL::DOUBLE, NULL::DOUBLE, NULL::DOUBLE, NULL::DOUBLE,
                           NULL::VARCHAR, NULL::VARCHAR, NULL::BIGINT, NULL::DOUBLE, NULL::DOUBLE))
      AS t(load_test_name, loadtest_id, thread_id, effective_username, iteration, action_index, source_label,
           think_time_basis, original_think_time_sec, think_time_scale, scaled_think_time_sec, applied_think_time_sec,
           sleep_start_utc, sleep_end_utc, queries_in_action, sleep_start_epoch, sleep_end_epoch)
    WHERE false
    """)

# One row per user action (page / interaction) and virtual user
con.execute("""
CREATE OR REPLACE VIEW actions AS
SELECT loadtest_id, model, level, thread_id, user_key, iteration, action_index,
       any_value(source_label)          AS source_label,
       count(*)                         AS n_queries,
       min(start_time)                  AS t_start,
       max(end_time)                    AS t_end,
       sum(duration)                    AS serial_time_s,     -- upper bound (queries run one after another)
       max(duration)                    AS slowest_query_s,   -- lower bound (ideal parallelism)
       max(end_time) - min(start_time)  AS action_wall_s,
       max(original_think_time_sec)     AS recorded_pause_s,
       max(applied_think_time_sec)      AS applied_pause_s
FROM results
GROUP BY ALL
""")

# Focus level of every model
lvl_sql = (f"SELECT DISTINCT model, CAST({int(FOCUS_LEVEL)} AS BIGINT) AS lvl FROM results" if FOCUS_LEVEL
           else "SELECT model, max(level) AS lvl FROM results GROUP BY model")
con.execute(f"CREATE OR REPLACE VIEW focus_levels AS {lvl_sql}")
con.execute("""
CREATE OR REPLACE VIEW results_focus AS
SELECT r.* FROM results r JOIN focus_levels f ON f.model = r.model AND f.lvl = r.level
""")
FOCUS = {m: int(l) for m, l in con.execute("SELECT model, lvl FROM focus_levels").fetchall()}

RESULTS = {}   # nn -> DataFrame, for persistence and conclusions
ERRORS = {}    # nn -> message, if an analysis failed (the notebook carries on)

def q(sql, key=None, show=True):
    df = con.execute(sql).fetchdf()
    if MASK_UPN:
        for c in [c for c in df.columns if c in ("user_key", "effective_username")]:
            df[c] = df[c].astype(str).map(lambda u: u if u.startswith("no_rls") or "@" not in u
                                          else "user_" + hashlib.sha1(u.encode()).hexdigest()[:6])
    if key: RESULTS[key] = df
    if show: display(df)
    return df

class analysis:
    """Context of every analysis: it never breaks Run all. An error is printed and stored in ERRORS."""
    def __init__(self, nn, title): self.nn, self.title = nn, title
    def __enter__(self):
        print(f"▶ {self.nn} · {self.title}"); return self
    def __exit__(self, etype, value, tb):
        if etype is None: return False
        ERRORS[self.nn] = f"{etype.__name__}: {value}"
        print(f"Error in {self.nn} · {self.title}: {etype.__name__}: {str(value)[:300]}")
        plt.close("all")
        return True
    def has_data(self, df, reason="the query returned no rows with the current filters"):
        if df is None or len(df) == 0:
            print(f"No data for {self.nn} · {self.title}: {reason}")
            return False
        return True

def at_focus(df, col="level"):
    """Rows of the focus level of every model."""
    return df[[FOCUS.get(m) == l for m, l in zip(df["model"], df[col])]]

# ---------------------------------------------------------------------------------------------------------------------
# Chart helpers (all of them tolerate empty DataFrames and null values)
# ---------------------------------------------------------------------------------------------------------------------
plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})
PALETTE = plt.rcParams["axes.prop_cycle"].by_key()["color"]
RED, GREEN, AMBER, GREY, BLUE = "#C0392B", "#2E8B57", "#E6A23C", "#9AA0A6", "#4C78A8"

def finish():
    plt.tight_layout(); plt.show()

def _hline(ax, y, text):
    ax.axhline(y, color=RED, ls="--", lw=1.2, zorder=0)
    ax.annotate(text, xy=(1, y), xycoords=("axes fraction", "data"), xytext=(-4, 3), textcoords="offset points",
                ha="right", va="bottom", color=RED, fontsize=8)

def _vline(ax, x, text):
    ax.axvline(x, color=RED, ls="--", lw=1.2, zorder=0)
    ax.annotate(text, xy=(x, 1), xycoords=("data", "axes fraction"), xytext=(3, -3), textcoords="offset points",
                ha="left", va="top", color=RED, fontsize=8)

def bar_grouped(ax, df, x, cols, labels=None, colors=None, hline=None, hline_txt="", ylabel="", logy=False):
    if df is None or df.empty: return
    xs = df[x].astype(str).tolist(); n = len(cols); w = 0.8 / n; idx = np.arange(len(xs))
    for i, c in enumerate(cols):
        ax.bar(idx + (i - (n - 1) / 2) * w, df[c].astype(float), w, label=(labels or cols)[i], color=(colors or PALETTE)[i % len(colors or PALETTE)])
    ax.set_xticks(idx); ax.set_xticklabels(xs); ax.set_ylabel(ylabel)
    if logy: ax.set_yscale("log")
    if hline is not None: _hline(ax, hline, hline_txt)
    ax.legend(fontsize=8)

def barh_sorted(ax, df, label, value, colors=None, cmap_col=None, cmap="YlOrRd", cbar_label="", vline=None, vline_txt="",
                xlabel="", annotate_col=None, annotate_fmt="{:.1f}", marker_col=None, marker_txt=""):
    """Horizontal bars: the first row of the DataFrame ends up at the top."""
    if df is None or df.empty: return
    d = df.iloc[::-1].reset_index(drop=True); y = np.arange(len(d)); vals = d[value].astype(float)
    if cmap_col:
        c = d[cmap_col].astype(float); norm = plt.Normalize(c.min(), c.max() if c.max() > c.min() else c.min() + 1)
        bar_colors = plt.get_cmap(cmap)(norm(c.values))
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        plt.colorbar(sm, ax=ax, label=cbar_label)
    elif colors is not None:
        bar_colors = list(colors)[::-1]
    else:
        bar_colors = BLUE
    ax.barh(y, vals, color=bar_colors)
    ax.set_yticks(y); ax.set_yticklabels(d[label].astype(str).str.slice(0, 48)); ax.set_xlabel(xlabel)
    if annotate_col:
        for i, v in enumerate(vals):
            if pd.notna(v): ax.text(v, i, " " + annotate_fmt.format(d[annotate_col].iloc[i]), va="center", fontsize=8)
    if marker_col:
        ax.scatter(d[marker_col].astype(float), y, marker="D", s=28, color="black", zorder=3, label=marker_txt)
        ax.legend(fontsize=8, loc="lower right")
    if vline is not None: _vline(ax, vline, vline_txt)

def heatmap(ax, piv, fmt="{:.2f}", cmap="YlOrRd", cbar_label=""):
    if piv is None or piv.empty: return
    data = piv.values.astype(float)
    im = ax.imshow(data, aspect="auto", cmap=cmap)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([str(c) for c in piv.columns])
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels([str(i)[:40] for i in piv.index], fontsize=8)
    ax.grid(False)
    vmax = np.nanmax(data) if np.isfinite(data).any() else 1
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if np.isfinite(data[i, j]):
                ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center", fontsize=7,
                        color="white" if data[i, j] > 0.6 * vmax else "black")
    plt.colorbar(im, ax=ax, label=cbar_label)

def stacked_100(ax, df, tick_labels, cols, names, colors, ylabel="% of total"):
    if df is None or df.empty: return
    v = df[cols].astype(float).clip(lower=0).fillna(0); tot = v.sum(axis=1).replace(0, np.nan)
    pct = v.div(tot, axis=0).fillna(0) * 100; x = np.arange(len(df)); base = np.zeros(len(df))
    for c, name, color in zip(cols, names, colors):
        ax.bar(x, pct[c], bottom=base, label=name, color=color)
        for i, val in enumerate(pct[c]):
            if val >= 6: ax.text(x[i], base[i] + val / 2, f"{val:.0f}%", ha="center", va="center", fontsize=7, color="white")
        base += pct[c].values
    ax.set_xticks(x); ax.set_xticklabels(list(tick_labels), fontsize=8); ax.set_ylim(0, 100); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, ncol=len(cols), loc="upper center", bbox_to_anchor=(0.5, -0.12))

def boxplot_by(ax, df, by, value, ylabel=""):
    if df is None or df.empty: return
    groups = [(k, g[value].dropna().values) for k, g in df.groupby(by) if g[value].notna().any()]
    if not groups: return
    ax.boxplot([g for _, g in groups], showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1)); ax.set_xticklabels([str(k) for k, _ in groups]); ax.set_ylabel(ylabel)

def scatter_sized(ax, df, x, y, size, label, xlabel="", ylabel="", logx=False, logy=False, hlines=(), top_label=10, rank=None):
    if df is None or df.empty: return
    d = df.copy(); t = d[size].astype(float); s = 30 + 170 * (t / t.max()) if t.max() > 0 else 60
    ax.scatter(d[x].astype(float), d[y].astype(float), s=s, alpha=0.65, color=BLUE, edgecolor="white")
    for val in hlines: ax.axhline(val, color=RED, ls="--", lw=1, zorder=0)
    if logx: ax.set_xscale("log")
    if logy: ax.set_yscale("log")
    for _, r in d.sort_values(rank or y, ascending=False).head(top_label).iterrows():
        ax.annotate(str(r[label])[:28], (float(r[x]), float(r[y])), xytext=(4, 4), textcoords="offset points", fontsize=7)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)

def hist(ax, series, xlabel="", bins=30):
    s = pd.Series(series).dropna()
    if s.empty: return
    ax.hist(s, bins=bins, color=BLUE, edgecolor="white"); ax.set_xlabel(xlabel); ax.set_ylabel("number of pauses")

def diverging_barh(ax, df, label, value, xlabel=""):
    if df is None or df.empty: return
    d = df.iloc[::-1].reset_index(drop=True); y = np.arange(len(d)); v = d[value].astype(float).fillna(0)
    ax.barh(y, v, color=[GREEN if x <= 0 else RED for x in v])
    ax.set_yticks(y); ax.set_yticklabels(d[label].astype(str).str.slice(0, 48)); ax.axvline(0, color="black", lw=0.8)
    for i, x in enumerate(v): ax.text(x, i, f" {x:+.0f}%", va="center", ha="left" if x >= 0 else "right", fontsize=8)
    ax.set_xlabel(xlabel)

def small_multiples(groups, draw, ncols=3, width=5.2, height=3.2, title=None):
    """groups = [(name, DataFrame)]; draw(ax, name, df). At most TIMELINE_MAX_RUNS charts."""
    groups = list(groups)[:TIMELINE_MAX_RUNS]
    if not groups: return
    n = len(groups); nc = min(ncols, n); nr = math.ceil(n / nc)
    fig, axs = plt.subplots(nr, nc, figsize=(width * nc, height * nr), squeeze=False)
    for ax, (name, g) in zip(axs.flat, groups): draw(ax, name, g)
    for ax in list(axs.flat)[n:]: ax.set_visible(False)
    if title: fig.suptitle(title, y=1.0)
    finish()
print("Setup ready: DuckDB", duckdb.__version__, "| status/error format:", HAS_STATUS, "| think times:", HAS_PAUSES)

### Initial check
What has been loaded: runs, models, levels, rows, failed queries, think times and offsets.

In [ ]:
# Initial check: what has been loaded
n_runs, n_models, n_levels, n_rows, n_users = con.execute(
    "SELECT count(DISTINCT loadtest_id), count(DISTINCT model), count(DISTINCT level), count(*), count(DISTINCT thread_id) FROM results_all").fetchone()
n_failed = con.execute("SELECT count(*) FROM results_all WHERE status <> 'ok'").fetchone()[0]
n_pauses, n_offsets = con.execute("SELECT count(*) FILTER (WHERE source_label <> 'initial_offset'), count(*) FILTER (WHERE source_label = 'initial_offset') FROM pauses").fetchone()
HAS_PAUSES = n_pauses > 0
HAS_OFFSETS = n_offsets > 0
print(f"Runs: {n_runs} | models: {n_models} | levels: {n_levels} | rows: {n_rows:,} (failed queries: {n_failed:,})")
print(f"Think times: {'yes' if HAS_PAUSES else 'no'} ({n_pauses:,} pauses) | staggered start (offset): {'yes' if HAS_OFFSETS else 'no'} ({n_offsets:,} users)")
print(f"Focus level per model: {FOCUS if FOCUS else 'no data'}")
if n_rows == 0:
    print("No data: the filters (SINCE_UTC, RUN_IDS, MODELS, LEVELS) leave no rows. Every analysis will print 'No data'.")
else:
    q("""
    SELECT r.model, r.level, r.loadtest_id AS run_id, count(DISTINCT r.thread_id) AS users, count(*) AS row_count,
           count(*) FILTER (WHERE r.status <> 'ok') AS failed,
           any_value(r.iterations) AS iterations, any_value(r.delay_sec) AS delay_sec,
           ts(min(r.start_time)) AS start_utc, round(max(r.end_time) - min(r.start_time), 1) AS duration_s,
           coalesce(any_value(p.n_pauses), 0) > 0 AS with_think_times, coalesce(any_value(p.n_offsets), 0) > 0 AS with_offset
    FROM results_all r
    LEFT JOIN (SELECT loadtest_id,
                      count(*) FILTER (WHERE source_label <> 'initial_offset') AS n_pauses,
                      count(*) FILTER (WHERE source_label = 'initial_offset')  AS n_offsets
               FROM pauses GROUP BY loadtest_id) p USING (loadtest_id)
    GROUP BY r.model, r.level, r.loadtest_id
    ORDER BY r.model, r.level, start_utc
    """, "INITIAL")

# Block A · Overall report performance

## Q01 · Executive summary per model and level
**What it measures:** KPIs of every run: users with results, queries, test duration, throughput (queries/s), mean and percentiles, % of queries within `APDEX_T` and Apdex.

**How it is computed:** Groups `results` by model, level and run. Percentiles with `quantile_cont` over `duration`; Apdex = (satisfied + 0.5 × tolerating) / total with T = `APDEX_T`. Failed queries (3.4+) are counted separately in `failed_queries` and excluded from latencies.

**How to read it:** p50 is the typical experience, p90/p95 the tail. `users_without_results` must be 0. An Apdex < 0.85 or a p95 above `APDEX_T` means the level is not sustained.

**Skew the test may introduce:** The test duration includes the start offset and the think times, so `queries_per_s` is the pace of the realistic scenario, not the maximum capacity of the model.

**Visual:** Table + grouped bars p50/p90/p95 per level (one figure per model) with a horizontal line at `APDEX_T`: the growth of the tail at a glance.

In [ ]:
with analysis("Q01", "Executive summary per model and level") as a:
    df = q(f"""
    WITH f AS (SELECT loadtest_id, count(*) AS failed FROM results_all WHERE status <> 'ok' GROUP BY loadtest_id)
    SELECT model, level, loadtest_id AS run_id,
           count(DISTINCT thread_id)                           AS users_with_results,
           level - count(DISTINCT thread_id)                   AS users_without_results,
           count(*)                                            AS queries,
           coalesce(any_value(f.failed), 0)                    AS failed_queries,
           round(max(end_time) - min(start_time), 1)           AS test_duration_s,
           round(count(*) / nullif(max(end_time) - min(start_time), 0), 2) AS queries_per_s,
           round(avg(duration), 3)                             AS mean_s,
           round(quantile_cont(duration, 0.50), 3)             AS p50_s,
           round(quantile_cont(duration, 0.90), 3)             AS p90_s,
           round(quantile_cont(duration, 0.95), 3)             AS p95_s,
           round(quantile_cont(duration, 0.99), 3)             AS p99_s,
           round(max(duration), 3)                             AS max_s,
           round(100.0 * avg(CASE WHEN duration <= {APDEX_T} THEN 1 ELSE 0 END), 1) AS pct_queries_within_T,
           round((count(*) FILTER (WHERE duration <= {APDEX_T})
                  + 0.5 * count(*) FILTER (WHERE duration > {APDEX_T} AND duration <= 4 * {APDEX_T})) / count(*), 3) AS apdex
    FROM results LEFT JOIN f USING (loadtest_id)
    GROUP BY model, level, loadtest_id
    ORDER BY model, level
    """, "Q01")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            fig, ax = plt.subplots(figsize=(12, 5))
            bar_grouped(ax, g.sort_values("level"), "level", ["p50_s", "p90_s", "p95_s"], ["p50", "p90", "p95"],
                        hline=APDEX_T, hline_txt=f"Apdex T = {APDEX_T:g} s", ylabel="seconds")
            ax.set_title(f"Q01 · {model}: query latency by number of users"); ax.set_xlabel("concurrent users (nominal level)")
            finish()

## Q02 · Degradation curve and knee point
**What it measures:** How latency grows when concurrency rises and at which level it breaks.

**How it is computed:** p50/p90/p95 per model and level; `p90_degradation_factor` = p90 / p90 of the base level. State: «Exceeds SLA» if p90 > `SLA_SEC`, «Knee» if the factor ≥ `KNEE_FACTOR`, otherwise «OK».

**How to read it:** A flat curve is a model that absorbs the load; the first level in «Knee» or «Exceeds SLA» marks the recommended user limit.

**Skew the test may introduce:** Levels run one after another without clearing the cache: the first level pays the cold cache and the following ones benefit from it, which can smooth the curve (see Q12).

**Visual:** Lines p50/p90/p95 against the level, one series per model (logarithmic X axis when the range is > 20×), SLA line and an annotation on the first problematic level.

In [ ]:
with analysis("Q02", "Degradation curve and knee point") as a:
    df = q(f"""
    WITH lv AS (
      SELECT model, level,
             quantile_cont(duration, 0.50) AS p50, quantile_cont(duration, 0.90) AS p90, quantile_cont(duration, 0.95) AS p95
      FROM results GROUP BY model, level),
    b AS (SELECT model, min(level) AS base_level, arg_min(p90, level) AS p90_base FROM lv GROUP BY model)
    SELECT lv.model, lv.level, round(p50, 3) AS p50_s, round(p90, 3) AS p90_s, round(p95, 3) AS p95_s,
           b.base_level, round(p90 / nullif(b.p90_base, 0), 2) AS p90_degradation_factor,
           CASE WHEN p90 > {SLA_SEC} THEN 'Exceeds SLA'
                WHEN p90 / nullif(b.p90_base, 0) >= {KNEE_FACTOR} THEN 'Knee'
                ELSE 'OK' END AS state
    FROM lv JOIN b USING (model)
    ORDER BY model, level
    """, "Q02")
    if a.has_data(df):
        fig, ax = plt.subplots(figsize=(12, 5))
        for i, (model, g) in enumerate(df.groupby("model")):
            g = g.sort_values("level"); c = PALETTE[i % len(PALETTE)]
            ax.plot(g.level, g.p50_s, ls=":", color=c, marker="o", ms=3, label=f"{model} p50")
            ax.plot(g.level, g.p90_s, ls="-", color=c, marker="o", label=f"{model} p90")
            ax.plot(g.level, g.p95_s, ls="--", color=c, marker="o", ms=3, label=f"{model} p95")
            for r in g[g.state != "OK"].head(1).itertuples():
                ax.annotate(f"{r.state}\n{r.level} users", (r.level, r.p90_s), xytext=(14, 18), textcoords="offset points",
                            arrowprops=dict(arrowstyle="->"), fontsize=8, color=RED)
        if df.level.max() / max(df.level.min(), 1) > 20:
            ax.set_xscale("log"); ax.set_xticks(sorted(df.level.unique())); ax.xaxis.set_major_formatter(plt.ScalarFormatter())
        _hline(ax, SLA_SEC, f"SLA = {SLA_SEC:g} s")
        ax.set_xlabel("concurrent users (nominal level)"); ax.set_ylabel("seconds"); ax.legend(fontsize=8, ncol=3)
        ax.set_title("Q02 · Latency degradation with concurrency"); finish()

## Q03 · Throughput versus latency
**What it measures:** Queries/s and actions/min served at every level next to the p90. Saturation is when throughput stops growing while the p90 rises.

**How it is computed:** queries/s = number of queries / (last end − first start); actions/min from the `actions` view; p90 of `duration`.

**How to read it:** Growing throughput with a stable p90 = spare capacity. Flat throughput with a growing p90 = saturation.

**Skew the test may introduce:** The window includes the offset and the think times: this is the throughput of the realistic scenario, not the maximum capacity of the model.

**Visual:** Combined: bars of queries/s (left axis) and a p90 line (right axis) per level, one chart per model.

In [ ]:
with analysis("Q03", "Throughput versus latency") as a:
    df = q("""
    WITH r AS (SELECT model, level, loadtest_id, count(*) AS n, min(start_time) AS t0, max(end_time) AS t1,
                      quantile_cont(duration, 0.90) AS p90
               FROM results GROUP BY ALL),
    a AS (SELECT loadtest_id, count(*) AS n_actions FROM actions GROUP BY loadtest_id)
    SELECT r.model, r.level, r.loadtest_id AS run_id,
           round(r.n / nullif(r.t1 - r.t0, 0), 2)                 AS queries_per_s,
           round(60.0 * a.n_actions / nullif(r.t1 - r.t0, 0), 1)  AS actions_per_min,
           round(r.p90, 3)                                        AS p90_s
    FROM r JOIN a USING (loadtest_id)
    ORDER BY r.model, r.level
    """, "Q03")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            g = g.sort_values("level"); fig, ax = plt.subplots(figsize=(12, 5))
            bar_grouped(ax, g, "level", ["queries_per_s"], ["queries/s"], ylabel="queries per second")
            ax2 = ax.twinx(); ax2.grid(False)
            ax2.plot(range(len(g)), g.p90_s, color=RED, marker="o", label="p90 (s)"); ax2.set_ylabel("p90 (s)"); ax2.legend(loc="upper right", fontsize=8)
            ax.set_xlabel("concurrent users (nominal level)"); ax.set_title(f"Q03 · {model}: throughput versus latency"); finish()

## Q04 · Timeline: active users, queries in flight and p90
**What it measures:** The evolution, in `BUCKET_SEC` intervals, of every run: users with an open session, queries running at the same time (average of the interval), queries started and p90 of the started ones.

**How it is computed:** Splits every run in intervals; active users = users whose first and last query overlap the interval; queries in flight = busy time inside the interval / `BUCKET_SEC`.

**How to read it:** The ramp-up must be staggered if there is `ramp_up_sec`; the plateau, flat; and the p90 should not shoot up when the plateau is reached.

**Skew the test may introduce:** It shows the ramp-up caused by the offset and the **dilution caused by think times**: 100 active users are not 100 queries in flight.

**Visual:** Small multiples of lines (one run per chart, the latest runs of the focus level, at most `TIMELINE_MAX_RUNS`): users and queries on the left axis, p90 on the right.

In [ ]:
with analysis("Q04", "Timeline: active users, queries in flight and p90") as a:
    recent = q(f"SELECT loadtest_id, min(start_time) AS t0 FROM results_focus GROUP BY loadtest_id ORDER BY t0 DESC LIMIT {int(TIMELINE_MAX_RUNS)}", show=False)
    if a.has_data(recent):
        id_list = ", ".join("'" + r.replace("'", "''") + "'" for r in recent.loadtest_id)
        con.execute(f"CREATE OR REPLACE VIEW results_q04 AS SELECT * FROM results WHERE loadtest_id IN ({id_list})")   # only the runs that are drawn
        df = q(f"""
        WITH bounds AS (
          SELECT loadtest_id, model, level, floor(min(start_time) / {BUCKET_SEC}) * {BUCKET_SEC} AS t0, max(end_time) AS t1
          FROM results_q04 GROUP BY ALL),
        bk AS (
          SELECT loadtest_id, model, level, t0 + k * {BUCKET_SEC} AS b0, t0 + (k + 1) * {BUCKET_SEC} AS b1
          FROM (SELECT *, unnest(range(0, CAST(ceil((t1 - t0) / {BUCKET_SEC}) AS BIGINT) + 1)) AS k FROM bounds)),
        us AS (SELECT loadtest_id, thread_id, min(start_time) AS s0, max(end_time) AS s1 FROM results_q04 GROUP BY ALL),
        ua AS (SELECT bk.loadtest_id, bk.b0, count(*) AS active_users
               FROM bk JOIN us ON us.loadtest_id = bk.loadtest_id AND us.s0 < bk.b1 AND us.s1 > bk.b0
               GROUP BY ALL),
        ov AS (SELECT bk.loadtest_id, bk.b0,
                      sum(greatest(0, least(r.end_time, bk.b1) - greatest(r.start_time, bk.b0))) / {BUCKET_SEC} AS in_flight
               FROM bk JOIN results_q04 r ON r.loadtest_id = bk.loadtest_id AND r.start_time < bk.b1 AND r.end_time > bk.b0
               GROUP BY ALL),
        st AS (SELECT loadtest_id, floor(start_time / {BUCKET_SEC}) * {BUCKET_SEC} AS b0,
                      count(*) AS started, quantile_cont(duration, 0.90) AS p90
               FROM results_q04 GROUP BY ALL)
        SELECT bk.model, bk.level, bk.loadtest_id AS run_id, ts(bk.b0) AS timestamp_utc,
               bk.b0 - min(bk.b0) OVER (PARTITION BY bk.loadtest_id) AS sec_from_start,
               coalesce(ua.active_users, 0)       AS active_users,
               round(coalesce(ov.in_flight, 0), 2) AS queries_in_flight,
               coalesce(st.started, 0)            AS queries_started,
               round(st.p90, 3)                   AS p90_s
        FROM bk
        LEFT JOIN ua USING (loadtest_id, b0)
        LEFT JOIN ov USING (loadtest_id, b0)
        LEFT JOIN st USING (loadtest_id, b0)
        ORDER BY run_id, sec_from_start
        """, "Q04", show=False)
        if a.has_data(df):
            print(f"{len(df)} intervals of {BUCKET_SEC} s in {df.run_id.nunique()} runs (the first 30 are shown):")
            display(df.head(30))
            def _draw(ax, name, g):
                ax.plot(g.sec_from_start, g.active_users, color=BLUE, label="active users")
                ax.plot(g.sec_from_start, g.queries_in_flight, color=GREEN, label="queries in flight")
                ax2 = ax.twinx(); ax2.grid(False); ax2.plot(g.sec_from_start, g.p90_s, color=RED, ls="--", lw=1, label="p90 (s)")
                ax.set_title(name.replace(f"{RUN_PREFIX}-", ""), fontsize=8); ax.set_xlabel("s from the start", fontsize=8); ax.legend(fontsize=6, loc="upper left")
            small_multiples(df.groupby("run_id"), _draw, title="Q04 · Active users, queries in flight (left) and p90 (right)")

# Block B · Performance per action (page / interaction) and journey

## Q05 · Time per user action (page, filter, slicer…)
**What it measures:** For every action of the JSON file: number of queries and perceived response time, with a serial bound and a parallel bound, and the % of actions within `SLA_SEC`.

**How it is computed:** `actions` view: serial bound = `sum(duration)` of the action; parallel bound = `max(duration)`. Percentiles per action and level; `pct_within_sla` = % with serial time ≤ `SLA_SEC`.

**How to read it:** Actions above the SLA are the ones users perceive as slow. If only the serial bound exceeds the SLA, the slowness comes from the serialisation of the tool.

**Skew the test may introduce:** The tool runs one after another the visuals that Power BI launches in parallel: the serial bound inflates actions with many queries; the parallel bound corrects it.

**Visual:** Horizontal bars sorted by serial p90 (focus level) with a marker for the parallel bound and a vertical SLA line.

In [ ]:
with analysis("Q05", "Time per user action") as a:
    df = q(f"""
    SELECT model, level, action_index, any_value(source_label) AS action,
           round(avg(n_queries), 1)                              AS queries_per_action,
           count(*)                                              AS executions,
           round(quantile_cont(serial_time_s, 0.50), 3)          AS p50_serial_s,
           round(quantile_cont(serial_time_s, 0.90), 3)          AS p90_serial_s,
           round(quantile_cont(slowest_query_s, 0.90), 3)        AS p90_parallel_bound_s,
           round(100.0 * avg(CASE WHEN serial_time_s <= {SLA_SEC} THEN 1 ELSE 0 END), 1) AS pct_within_sla
    FROM actions
    GROUP BY model, level, action_index
    ORDER BY model, level, action_index
    """, "Q05")
    focus = at_focus(df) if len(df) else df
    if a.has_data(focus):
        for model, g in focus.groupby("model"):
            g = g.sort_values("p90_serial_s", ascending=False).copy()
            g["label"] = g.action_index.astype(str) + " · " + g.action.astype(str)
            fig, ax = plt.subplots(figsize=(12, max(3.5, 0.5 * len(g) + 1.8)))
            barh_sorted(ax, g, "label", "p90_serial_s", vline=SLA_SEC, vline_txt=f"SLA = {SLA_SEC:g} s", xlabel="p90 of the action time (s)",
                        marker_col="p90_parallel_bound_s", marker_txt="parallel bound (p90)", annotate_col="pct_within_sla", annotate_fmt="{:.0f}% within SLA")
            ax.set_title(f"Q05 · {model} ({int(g.level.iloc[0])} users): time per action, serial and parallel bound"); finish()

## Q06 · Full journey per user and iteration
**What it measures:** The total time a user spends waiting for the model across the whole report journey, the wall-clock time of the journey and the share that is query waiting.

**How it is computed:** Per user and iteration: `sum(duration)`, `max(end_time) − min(start_time)` and their ratio.

**How to read it:** Compare the accumulated wait between levels: if the box rises with the load, the whole journey gets worse even if every query looks acceptable.

**Skew the test may introduce:** The wall-clock time includes think times; the waiting share is low by design. Look at `query_time_s`, not at the wall clock.

**Visual:** Box plot of `query_time_s` per level (one chart per model): median, dispersion and extreme journeys.

In [ ]:
with analysis("Q06", "Full journey per user and iteration") as a:
    df = q("""
    WITH j AS (
      SELECT model, level, loadtest_id, thread_id, iteration,
             count(*) AS queries, sum(duration) AS query_time_s,
             max(end_time) - min(start_time) AS wall_time_s
      FROM results GROUP BY ALL)
    SELECT model, level, count(*) AS journeys,
           round(avg(query_time_s), 2)                    AS mean_query_time_s,
           round(quantile_cont(query_time_s, 0.5), 2)     AS p50_query_time_s,
           round(quantile_cont(query_time_s, 0.9), 2)     AS p90_query_time_s,
           round(avg(wall_time_s), 1)                     AS mean_wall_s,
           round(100.0 * avg(query_time_s / nullif(wall_time_s, 0)), 1) AS pct_wall_in_queries
    FROM j GROUP BY model, level ORDER BY model, level
    """, "Q06")
    if a.has_data(df):
        detail = q("""
        SELECT model, level, loadtest_id, thread_id, iteration, sum(duration) AS query_time_s
        FROM results GROUP BY ALL
        """, show=False)
        for model, g in detail.groupby("model"):
            fig, ax = plt.subplots(figsize=(12, 5))
            boxplot_by(ax, g, "level", "query_time_s", ylabel="accumulated query time per journey (s)")
            ax.set_xlabel("concurrent users (nominal level)"); ax.set_title(f"Q06 · {model}: total wait for the model across the report journey"); finish()

# Block C · Performance per visual

## Q07 · Ranking of the slowest visuals
**What it measures:** Per visual: executions, mean, p50/p90/p95, maximum, mean rows and the **% of the total query time** it consumes (where to optimise first).

**How it is computed:** Over `results_focus` (focus level of every model), grouped by `visual_label` (title + first 5 characters of the id).

**How to read it:** Optimise first the visuals with a high p90 **and** a large % of the total time. A visual that concentrates > 25 % is a bottleneck.

**Skew the test may introduce:** None in particular; it is the highest load level, so it reflects the worst case measured.

**Visual:** Horizontal bars of the `TOP_N` by p90, coloured by % of total time and annotated.

In [ ]:
with analysis("Q07", "Ranking of the slowest visuals") as a:
    df = q(f"""
    WITH v AS (
      SELECT model, visual_label, count(*) AS executions,
             avg(duration) AS mean, quantile_cont(duration, 0.5) AS p50, quantile_cont(duration, 0.9) AS p90,
             quantile_cont(duration, 0.95) AS p95, max(duration) AS maximum, sum(duration) AS total, avg(row_count) AS mean_rows
      FROM results_focus GROUP BY model, visual_label)
    SELECT model, visual_label, executions, round(mean, 3) AS mean_s, round(p50, 3) AS p50_s, round(p90, 3) AS p90_s,
           round(p95, 3) AS p95_s, round(maximum, 3) AS max_s, round(mean_rows, 0) AS mean_rows,
           round(100.0 * total / sum(total) OVER (PARTITION BY model), 1) AS pct_total_time
    FROM v
    QUALIFY row_number() OVER (PARTITION BY model ORDER BY p90 DESC) <= {TOP_N}
    ORDER BY model, p90_s DESC
    """, "Q07")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            fig, ax = plt.subplots(figsize=(12, max(3.5, 0.45 * len(g) + 1.8)))
            barh_sorted(ax, g, "visual_label", "p90_s", cmap_col="pct_total_time", cbar_label="% of the total query time",
                        annotate_col="pct_total_time", annotate_fmt="{:.0f}% of the time", xlabel="p90 (s)")
            ax.set_title(f"Q07 · {model}: slowest visuals (focus level, {FOCUS.get(model)} users)"); finish()

## Q08 · Heat map visual × concurrency level
**What it measures:** The p90 of every visual at every level and its degradation factor against the base level: which visuals break under load.

**How it is computed:** p90 per visual and level; `factor_vs_base` = p90 / p90 of the lowest level.

**How to read it:** Bands that get darker to the right are visuals sensitive to concurrency. A visual that is slow at every level is a calculation problem, not a contention problem.

**Skew the test may introduce:** The base level already comes with a cold cache and the rest with a warm one (Q12): a factor < 1 can be the cache.

**Visual:** Heat map (rows = `TOP_N` visuals by maximum p90, columns = levels) with the p90 annotated and, next to it, the degradation factor.

In [ ]:
with analysis("Q08", "Heat map visual × concurrency level") as a:
    df = q("""
    WITH v AS (SELECT model, visual_label, level, quantile_cont(duration, 0.9) AS p90
               FROM results GROUP BY ALL),
    b AS (SELECT model, visual_label, arg_min(p90, level) AS p90_base FROM v GROUP BY ALL)
    SELECT v.model, v.visual_label, v.level, round(v.p90, 3) AS p90_s,
           round(v.p90 / nullif(b.p90_base, 0), 2) AS factor_vs_base
    FROM v JOIN b USING (model, visual_label)
    ORDER BY v.model, v.visual_label, v.level
    """, "Q08", show=False)
    if a.has_data(df):
        print(f"{len(df)} visual × level combinations (the first 30 are shown):"); display(df.head(30))
        for model, g in df.groupby("model"):
            piv = g.pivot_table(index="visual_label", columns="level", values="p90_s")
            top = piv.max(axis=1).sort_values(ascending=False).head(int(TOP_N)).index
            piv = piv.loc[top]; fac = g.pivot_table(index="visual_label", columns="level", values="factor_vs_base").loc[top]
            fig, axs = plt.subplots(1, 2, figsize=(16, max(3.5, 0.42 * len(piv) + 1.8)))
            heatmap(axs[0], piv, fmt="{:.2f}", cbar_label="p90 (s)"); axs[0].set_title("p90 (s)")
            heatmap(axs[1], fac, fmt="{:.1f}×", cmap="PuBu", cbar_label="factor versus the base level"); axs[1].set_title("degradation factor")
            for ax in axs: ax.set_xlabel("concurrent users (nominal level)")
            fig.suptitle(f"Q08 · {model}: p90 per visual and level"); finish()

## Q09 · Stability of every visual (variability)
**What it measures:** The coefficient of variation (standard deviation / mean) and the p99/p50 ratio of every visual at the focus level. A visual that is fast on average but has a high CV produces an erratic experience.

**How it is computed:** `stddev_samp(duration) / avg(duration)` and `p99 / p50`, with at least 5 executions per visual.

**How to read it:** CV < 0.5: stable; 0.5–1: variable; > 1: erratic (usually contention, cache or a different volume of rows per RLS identity).

**Skew the test may introduce:** With RLS, the cache and the volume depend on the identity: part of the variability comes from there (see Q11).

**Visual:** Scatter of the mean (X) against the CV (Y), size = executions, labels on the 10 with the highest CV and guide lines at 0.5 and 1.

In [ ]:
with analysis("Q09", "Stability of every visual (variability)") as a:
    df = q("""
    SELECT model, visual_label, count(*) AS executions,
           round(avg(duration), 3) AS mean_s,
           round(stddev_samp(duration) / nullif(avg(duration), 0), 2) AS cv,
           round(quantile_cont(duration, 0.99) / nullif(quantile_cont(duration, 0.5), 0), 2) AS p99_p50_ratio
    FROM results_focus
    GROUP BY model, visual_label
    HAVING count(*) >= 5
    ORDER BY cv DESC
    """, "Q09")
    if a.has_data(df, "no visual reaches 5 executions at the focus level"):
        for model, g in df.groupby("model"):
            fig, ax = plt.subplots(figsize=(12, 5))
            scatter_sized(ax, g.dropna(subset=["cv"]), "mean_s", "cv", "executions", "visual_label",
                          xlabel="mean duration (s)", ylabel="coefficient of variation", hlines=(0.5, 1.0))
            ax.set_title(f"Q09 · {model}: mean versus variability (guide lines at CV = 0.5 and 1)"); finish()

## Q10 · Volume returned versus time
**What it measures:** Whether the slowness of a visual is explained by the number of rows it returns (big tables and matrices, transfer) or by DAX calculation.

**How it is computed:** Mean rows and p50 per visual at the focus level, and the rows–duration correlation per model (`corr`).

**How to read it:** High correlation: the cost is volume (limit rows, paginate, aggregate). Low correlation with slow visuals: the cost is calculation (measures, relationships).

**Skew the test may introduce:** `duration` includes the reading of every row by the test client, which a real report does not always do.

**Visual:** Log-log scatter of mean rows (X) against p50 (Y) per visual, size = executions, and the correlation in the title.

In [ ]:
with analysis("Q10", "Volume returned versus time") as a:
    df = q("""
    WITH c AS (SELECT model, corr(row_count, duration) AS model_correlation FROM results_focus GROUP BY model),
    v AS (SELECT model, visual_label, count(*) AS executions,
                 avg(row_count) AS mean_rows, quantile_cont(duration, 0.5) AS p50
          FROM results_focus GROUP BY model, visual_label)
    SELECT v.model, v.visual_label, v.executions, round(v.mean_rows, 0) AS mean_rows,
           round(v.p50, 3) AS p50_s, round(c.model_correlation, 2) AS model_correlation
    FROM v JOIN c USING (model)
    ORDER BY v.model, p50_s DESC
    """, "Q10")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            g = g.assign(mean_rows=g.mean_rows.clip(lower=1), p50_s=g.p50_s.clip(lower=0.001))
            fig, ax = plt.subplots(figsize=(12, 5))
            scatter_sized(ax, g, "mean_rows", "p50_s", "executions", "visual_label", xlabel="rows returned (mean, log scale)",
                          ylabel="p50 (s, log scale)", logx=True, logy=True)
            ax.set_title(f"Q10 · {model}: rows versus time (correlation = {g.model_correlation.iloc[0]})"); finish()

# Block D · Performance per user (RLS identity)

## Q11 · Performance per virtual user / RLS identity
**What it measures:** All users run the same scenario; the differences are due to security (volume of data visible through RLS) or to contention. It flags «Atypical» when |z| ≥ 2.

**How it is computed:** Per identity at the focus level: queries, mean, p90, total time and mean rows; z-score of the mean against the mean of the level.

**How to read it:** A slow atypical identity usually has a costlier RLS role or much more visible data. These are test identities: the analysis is about the model and its security, not about people.

**Skew the test may introduce:** Without RLS every virtual user is its own identity (`no_rls_thread_N`) and the differences are just noise and contention.

**Visual:** Horizontal bars of the p90 of the `TOP_N` slowest (red if atypical, line with the p90 of the level) and, next to it, mean rows against mean duration.

In [ ]:
with analysis("Q11", "Performance per virtual user / RLS identity") as a:
    df = q("""
    WITH u AS (
      SELECT model, level, user_key, count(DISTINCT thread_id) AS threads, count(*) AS queries,
             avg(duration) AS mean, quantile_cont(duration, 0.9) AS p90, sum(duration) AS total, avg(row_count) AS mean_rows
      FROM results_focus GROUP BY ALL),
    l AS (SELECT model, level, avg(mean) AS m, stddev_samp(mean) AS sd FROM u GROUP BY ALL)
    SELECT u.model, u.level, u.user_key, u.threads, u.queries,
           round(u.mean, 3) AS mean_s, round(u.p90, 3) AS p90_s, round(u.total, 1) AS total_s,
           round(u.mean_rows, 0) AS mean_rows,
           round((u.mean - l.m) / nullif(l.sd, 0), 2) AS z_score,
           CASE WHEN abs((u.mean - l.m) / nullif(l.sd, 0)) >= 2 THEN 'Atypical' ELSE 'Normal' END AS state
    FROM u JOIN l USING (model, level)
    ORDER BY u.model, p90_s DESC
    """, "Q11")
    if a.has_data(df):
        level_p90 = q("SELECT model, quantile_cont(duration, 0.9) AS p90 FROM results_focus GROUP BY model", show=False).set_index("model").p90.to_dict()
        for model, g in df.groupby("model"):
            top = g.sort_values("p90_s", ascending=False).head(int(TOP_N))
            fig, axs = plt.subplots(1, 2, figsize=(15, max(3.5, 0.4 * len(top) + 2)))
            barh_sorted(axs[0], top, "user_key", "p90_s", colors=[RED if s == "Atypical" else BLUE for s in top.state],
                        vline=level_p90.get(model), vline_txt="p90 of the level", xlabel="p90 (s)")
            axs[0].set_title("p90 per identity (red = atypical)")
            scatter_sized(axs[1], g, "mean_rows", "mean_s", "queries", "user_key", xlabel="mean rows returned", ylabel="mean duration (s)", rank="mean_s")
            axs[1].set_title("visible rows versus duration")
            fig.suptitle(f"Q11 · {model} ({FOCUS.get(model)} users)"); finish()

## Q12 · Cold cache versus warm cache
**What it measures:** The difference between the first execution of every DAX query in the test (cold cache) and its repetitions, and between iteration 0 and the following ones. It quantifies how much the cache helps the high levels.

**How it is computed:** `row_number()` per run and query hash: 1 = cold, > 1 = warm. Percentiles of each group.

**How to read it:** A big cold/warm difference means the results depend on the cache: a test with a cold cache (after a refresh or a restart) will be worse than the measured one.

**Skew the test may introduce:** Levels run one after another without clearing the cache: the first level warms up the following ones. With RLS the cache depends on the identity.

**Visual:** Grouped bars of p50 and p90, cold versus warm, per level.

In [ ]:
with analysis("Q12", "Cold cache versus warm cache") as a:
    # Variant A: per DAX query in the run. Variant B (optional): PARTITION BY loadtest_id, query_hash, user_key
    df = q("""
    WITH r AS (SELECT *, row_number() OVER (PARTITION BY loadtest_id, query_hash ORDER BY start_time) AS n_exec FROM results)
    SELECT model, level,
           CASE WHEN n_exec = 1 THEN 'cold (first execution)' ELSE 'warm (repeated)' END AS cache_state,
           CASE WHEN iteration = 0 THEN 'iteration 0' ELSE 'iteration >0' END AS iteration_group,
           count(*) AS queries,
           round(quantile_cont(duration, 0.5), 3) AS p50_s, round(quantile_cont(duration, 0.9), 3) AS p90_s
    FROM r
    GROUP BY ALL
    ORDER BY model, level, cache_state, iteration_group
    """, "Q12")
    if a.has_data(df):
        chart = q("""
        WITH r AS (SELECT *, row_number() OVER (PARTITION BY loadtest_id, query_hash ORDER BY start_time) AS n_exec FROM results)
        SELECT model, level, CASE WHEN n_exec = 1 THEN 'cold' ELSE 'warm' END AS cache,
               quantile_cont(duration, 0.5) AS p50_s, quantile_cont(duration, 0.9) AS p90_s
        FROM r GROUP BY ALL ORDER BY model, level
        """, show=False)
        for model, g in chart.groupby("model"):
            levels = sorted(g.level.unique()); w = pd.DataFrame({"level": levels}).set_index("level")
            for c in ("cold", "warm"):
                s = g[g.cache == c].set_index("level")
                w[f"{c} p50"] = s.p50_s.reindex(w.index); w[f"{c} p90"] = s.p90_s.reindex(w.index)
            fig, ax = plt.subplots(figsize=(12, 5))
            bar_grouped(ax, w.reset_index(), "level", ["cold p50", "warm p50", "cold p90", "warm p90"],
                        colors=["#9ECAE1", BLUE, "#FDC086", "#F58518"], ylabel="seconds")
            ax.set_xlabel("concurrent users (nominal level)"); ax.set_title(f"Q12 · {model}: cold versus warm cache"); finish()

# Block E · Skew introduced by the offset and the think times

## Q13 · Wall-clock breakdown of every virtual user
**What it measures:** Where the time of a virtual user goes, from the moment its notebook starts until its last query: initial offset + queries + think times + `delay_sec` waits + unexplained overhead.

**How it is computed:** Wall clock = last query − notebook start. Offset and start come from the `initial_offset` row; `delay_sec` waits are computed depending on whether think times are on (engine rules); the overhead is the difference and includes `sleep` imprecision, Python, ADOMD and the network.

**How to read it:** The overhead must be small (< 5 % of the wall clock). If it grows, the tool (not the model) is delaying the users.

**Skew the test may introduce:** This is the key analysis to quantify the total skew the test introduces. Without think times there are no pauses between actions and the wall clock is made almost only of queries and `delay_sec`.

**Visual:** 100 % stacked bars per model and level with fixed colours (offset grey, queries blue, think times green, delay amber, overhead red).

In [ ]:
with analysis("Q13", "Wall-clock breakdown of every virtual user") as a:
    df = q("""
    WITH qq AS (
      SELECT loadtest_id, model, level, thread_id,
             any_value(delay_sec) AS delay_sec, any_value(iterations) AS iterations_n,
             count(*) AS n, min(start_time) AS t_first, max(end_time) AS t_last,
             sum(duration) AS queries_s, sum(applied_think_time_sec) AS think_s
      FROM results GROUP BY ALL),
    o AS (SELECT loadtest_id, thread_id, max(applied_think_time_sec) AS offset_s, min(sleep_start_epoch) AS t_launch
          FROM pauses WHERE source_label = 'initial_offset' GROUP BY ALL),
    tt AS (SELECT DISTINCT loadtest_id FROM pauses WHERE source_label <> 'initial_offset'),
    d AS (
      SELECT qq.*, coalesce(o.offset_s, 0) AS offset_s, coalesce(o.t_launch, qq.t_first) AS t_ini,
             CASE WHEN tt.loadtest_id IS NOT NULL THEN qq.delay_sec * (qq.iterations_n - 1)          -- think times on
                  ELSE qq.delay_sec * (qq.n + qq.iterations_n - 2) END AS delay_waits_s              -- think times off
      FROM qq LEFT JOIN o USING (loadtest_id, thread_id) LEFT JOIN tt USING (loadtest_id))
    SELECT model, level, count(*) AS users,
           round(avg(t_last - t_ini), 1)       AS wall_s,
           round(avg(offset_s), 1)             AS initial_offset_s,
           round(avg(queries_s), 1)            AS queries_s,
           round(avg(think_s), 1)              AS think_time_s,
           round(avg(delay_waits_s), 1)        AS delay_waits_s,
           round(avg((t_last - t_ini) - offset_s - queries_s - think_s - delay_waits_s), 2) AS unexplained_overhead_s,
           round(100.0 * avg(queries_s / nullif(t_last - t_ini, 0)), 1) AS pct_wall_in_queries
    FROM d GROUP BY model, level ORDER BY model, level
    """, "Q13")
    if a.has_data(df):
        df = df.sort_values(["model", "level"]); fig, ax = plt.subplots(figsize=(max(10, 1.1 * len(df) + 4), 5.5))
        stacked_100(ax, df, [f"{m}\n{l} users" for m, l in zip(df.model, df.level)],
                    ["initial_offset_s", "queries_s", "think_time_s", "delay_waits_s", "unexplained_overhead_s"],
                    ["initial offset", "queries", "think times", "delay_sec waits", "unexplained overhead"],
                    [GREY, BLUE, GREEN, AMBER, RED])
        ax.set_title("Q13 · Where does the wall-clock time of a virtual user go?"); finish()

## Q14 · Think-time accuracy (scheduled versus actual)
**What it measures:** Sleep error, latency between the end of the pause and the query, actual gap measured between actions versus the applied think time, pauses capped by `max_think_time_sec` and the effect of jitter.

**How it is computed:** Joins `pauses` with `actions`: error = (sleep end − sleep start) − applied; actual gap = start of the action − end of the previous one.

**How to read it:** The actual gap must match the applied one (mean offset < 0.5 s). Capped pauses mean the scenario is faster than the recording.

**Skew the test may introduce:** If the actual gap is systematically above the applied one, the pace of the test is slower than the recorded one: less real pressure on the model.

**Visual:** Two charts: histogram of the sleep error (ms) and scatter of the applied think time (X) against the actual gap measured (Y) with the y = x diagonal.

In [ ]:
_Q14_CTE = r"""WITH g AS (
  SELECT loadtest_id, thread_id, iteration, action_index, t_start,
         t_start - lag(t_end) OVER (PARTITION BY loadtest_id, thread_id, iteration ORDER BY t_start) AS actual_gap_s
  FROM actions),
p AS (
  SELECT p.*, g.actual_gap_s, g.t_start,
         (p.sleep_end_epoch - p.sleep_start_epoch) - p.applied_think_time_sec AS sleep_error_s,
         g.t_start - p.sleep_end_epoch AS wake_to_query_s
  FROM pauses p
  LEFT JOIN g USING (loadtest_id, thread_id, iteration, action_index)
  WHERE p.source_label <> 'initial_offset')
"""
with analysis("Q14", "Think-time accuracy (scheduled versus actual)") as a:
    df = q(_Q14_CTE + """
    SELECT loadtest_id AS run_id, count(*) AS pauses,
           round(1000 * avg(sleep_error_s), 1)                     AS mean_sleep_error_ms,
           round(1000 * max(sleep_error_s), 1)                     AS max_sleep_error_ms,
           round(1000 * avg(wake_to_query_s), 1)                   AS wake_to_query_ms,
           round(avg(actual_gap_s - applied_think_time_sec), 3)    AS mean_gap_offset_s,
           round(quantile_cont(actual_gap_s - applied_think_time_sec, 0.95), 3) AS p95_gap_offset_s,
           count(*) FILTER (WHERE original_think_time_sec * think_time_scale > scaled_think_time_sec + 0.001) AS pauses_capped,
           round(sum(original_think_time_sec * think_time_scale - scaled_think_time_sec), 1) AS seconds_lost_to_cap,
           round(avg(applied_think_time_sec / nullif(scaled_think_time_sec, 0)), 3) AS mean_jitter_ratio,
           round(sum(applied_think_time_sec) / nullif(sum(original_think_time_sec), 0), 3) AS applied_vs_recorded_ratio
    FROM p GROUP BY loadtest_id ORDER BY loadtest_id
    """, "Q14")
    if a.has_data(df, "the selected runs have no think times (enable_think_times = false)"):
        detail = q(_Q14_CTE + "SELECT loadtest_id, applied_think_time_sec, actual_gap_s, sleep_error_s FROM p", show=False)
        fig, axs = plt.subplots(1, 2, figsize=(14, 5))
        hist(axs[0], detail.sleep_error_s * 1000, "sleep error in ms (actual − scheduled)")
        axs[0].set_title("Sleep error of the pauses")
        sub = detail.dropna(subset=["actual_gap_s"])
        if len(sub) > 20000: sub = sub.sample(20000, random_state=1)
        axs[1].scatter(sub.applied_think_time_sec, sub.actual_gap_s, s=8, alpha=0.4, color=BLUE)
        if len(sub):
            lim = float(max(sub.applied_think_time_sec.max(), sub.actual_gap_s.max())); axs[1].plot([0, lim], [0, lim], color=RED, ls="--", label="y = x")
            axs[1].legend(fontsize=8)
        axs[1].set_xlabel("applied think time (s)"); axs[1].set_ylabel("actual gap measured between actions (s)"); axs[1].set_title("Applied pause versus actual gap")
        finish()

## Q15 · Skew of the staggered start (offset) per virtual user
**What it measures:** For every virtual user: scheduled offset, notebook launch delay (cost of `runMultiple`), actual start of its first query and deviation from the schedule.

**How it is computed:** `sleep_start` of the `initial_offset` row ≈ the instant the notebook started; actual start = first query − first start of the run.

**How to read it:** If the actual start stretches far beyond `ramp_up_sec`, the nominal level takes longer to be reached (or is not reached: see Q17).

**Skew the test may introduce:** Without a configured offset, the analysis still shows the start dispersion inherent to `runMultiple`.

**Visual:** Small multiples (one run per chart): X = `thread_id`, dots = actual start, line = scheduled offset; the vertical distance is the skew.

In [ ]:
with analysis("Q15", "Skew of the staggered start (offset) per virtual user") as a:
    df = q("""
    WITH u AS (SELECT loadtest_id, model, level, thread_id, min(start_time) AS t_first FROM results GROUP BY ALL),
    o AS (SELECT loadtest_id, thread_id, applied_think_time_sec AS offset_s,
                 sleep_start_epoch AS t_launch, sleep_end_epoch AS t_wake
          FROM pauses WHERE source_label = 'initial_offset'),
    x AS (SELECT u.*, o.offset_s, o.t_launch, o.t_wake,
                 min(coalesce(o.t_launch, u.t_first)) OVER (PARTITION BY u.loadtest_id) AS t_run0
          FROM u LEFT JOIN o USING (loadtest_id, thread_id))
    SELECT model, level, loadtest_id AS run_id, thread_id,
           round(coalesce(offset_s, 0), 2)                        AS scheduled_offset_s,
           round(t_launch - t_run0, 2)                            AS notebook_launch_delay_s,
           round(t_first - t_run0, 2)                             AS actual_start_s,
           round((t_first - t_run0) - coalesce(offset_s, 0), 2)   AS skew_vs_scheduled_s,
           round(t_first - coalesce(t_wake, t_first), 3)          AS wake_to_first_query_s
    FROM x ORDER BY run_id, thread_id
    """, "Q15", show=False)
    if a.has_data(df):
        summary = df.groupby(["model", "level", "run_id"]).agg(
            users=("thread_id", "count"), scheduled_window_s=("scheduled_offset_s", "max"), actual_start_up_to_s=("actual_start_s", "max"),
            mean_skew_s=("skew_vs_scheduled_s", "mean"), p95_skew_s=("skew_vs_scheduled_s", lambda s: s.quantile(0.95))).round(2).reset_index()
        print("Summary per run (scheduled start window versus actual):"); display(summary)
        print("Detail per virtual user (the first 30):"); display(df.head(30))
        focus = at_focus(df)
        def _draw(ax, name, g):
            g = g.sort_values("thread_id")
            ax.step(g.thread_id, g.scheduled_offset_s, where="mid", color=RED, label="scheduled offset")
            ax.scatter(g.thread_id, g.actual_start_s, s=14, color=BLUE, label="actual start (first query)")
            ax.set_title(name.replace(f"{RUN_PREFIX}-", ""), fontsize=8); ax.set_xlabel("thread_id", fontsize=8); ax.set_ylabel("s from the first start", fontsize=8); ax.legend(fontsize=6)
        small_multiples(focus.groupby("run_id") if len(focus) else df.groupby("run_id"), _draw, title="Q15 · Scheduled offset versus actual start")

## Q16 · Nominal versus effective concurrency (think-time dilution)
**What it measures:** The nominal level (N users) against the peak of users with an open session, the peak of simultaneous queries, the mean of queries in flight (Little's law) and the duty cycle (% of the time a user is querying).

**How it is computed:** Event sweep (+1 at start and −1 at end) for the peaks; mean in flight = `sum(duration)` / duration; duty cycle = mean in flight / N.

**How to read it:** With realistic think times, 100 users can be equivalent to 3–10 simultaneous queries: that is the figure to communicate to the customer. A duty cycle < 10 % means a lot of dilution.

**Skew the test may introduce:** It is the direct effect of think times and of the offset on the real pressure the model receives.

**Visual:** Grouped bars per level (nominal, peak users, peak queries, mean in flight; log axis when the range is large) and the duty-cycle line (%) on the secondary axis.

In [ ]:
with analysis("Q16", "Nominal versus effective concurrency (think-time dilution)") as a:
    df = q("""
    WITH u AS (SELECT loadtest_id, thread_id, min(start_time) AS s0, max(end_time) AS s1 FROM results GROUP BY ALL),
    eu AS (SELECT loadtest_id, s0 AS t, 1 AS d FROM u UNION ALL SELECT loadtest_id, s1, -1 FROM u),
    cu AS (SELECT loadtest_id, max(c) AS peak_users FROM (
             SELECT loadtest_id, sum(d) OVER (PARTITION BY loadtest_id ORDER BY t, d ROWS UNBOUNDED PRECEDING) AS c FROM eu) x
           GROUP BY loadtest_id),
    eq AS (SELECT loadtest_id, start_time AS t, 1 AS d FROM results UNION ALL SELECT loadtest_id, end_time, -1 FROM results),
    cq AS (SELECT loadtest_id, max(c) AS peak_queries FROM (
             SELECT loadtest_id, sum(d) OVER (PARTITION BY loadtest_id ORDER BY t, d ROWS UNBOUNDED PRECEDING) AS c FROM eq) y
           GROUP BY loadtest_id),
    r AS (SELECT loadtest_id, model, level, sum(duration) AS busy, min(start_time) AS t0, max(end_time) AS t1
          FROM results GROUP BY ALL)
    SELECT r.model, r.level AS nominal_level, r.loadtest_id AS run_id,
           cu.peak_users, cq.peak_queries,
           round(r.busy / nullif(r.t1 - r.t0, 0), 2) AS mean_queries_in_flight,
           round(100.0 * r.busy / nullif(r.t1 - r.t0, 0) / r.level, 1) AS duty_cycle_pct
    FROM r JOIN cu USING (loadtest_id) JOIN cq USING (loadtest_id)
    ORDER BY r.model, r.level
    """, "Q16")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            g = g.sort_values("nominal_level"); fig, ax = plt.subplots(figsize=(12, 5))
            cols = ["nominal_level", "peak_users", "peak_queries", "mean_queries_in_flight"]
            wide = g[cols].astype(float)
            bar_grouped(ax, g, "nominal_level", cols, ["nominal level", "peak of active users", "peak of simultaneous queries", "mean queries in flight"],
                        colors=[GREY, BLUE, GREEN, AMBER], ylabel="users / queries", logy=bool(wide.max().max() / max(wide.replace(0, np.nan).min().min(), 0.1) > 50))
            ax2 = ax.twinx(); ax2.grid(False)
            ax2.plot(range(len(g)), g.duty_cycle_pct, color=RED, marker="o", label="duty cycle (%)"); ax2.set_ylabel("duty cycle (%)"); ax2.legend(loc="upper right", fontsize=8)
            ax.set_xlabel("concurrent users (nominal level)"); ax.set_title(f"Q16 · {model}: nominal versus effective concurrency"); finish()

## Q17 · Ramp phases: ramp-up, plateau and ramp-down
**What it measures:** Splits every run in ramp-up (users are still starting), plateau (all active) and ramp-down, and compares latency per phase. Only the plateau represents the nominal level.

**How it is computed:** Plateau = from the last user start to the first user end; if the plateau start is later than its end, there is no plateau (level not reached).

**How to read it:** A short plateau, or one with < 50 % of the queries, means the nominal level was hardly sustained: the measurement of that level is poorly representative.

**Skew the test may introduce:** A long offset compared with the duration of the journey can prevent the nominal level from being reached.

**Visual:** Grouped bars of p90 per phase and level and a table per run with `plateau_s` and the % of queries on the plateau.

In [ ]:
with analysis("Q17", "Ramp phases: ramp-up, plateau and ramp-down") as a:
    df = q("""
    WITH u AS (SELECT loadtest_id, thread_id, min(start_time) AS s0, max(end_time) AS s1 FROM results GROUP BY ALL),
    m AS (SELECT loadtest_id, max(s0) AS plateau_start, min(s1) AS plateau_end FROM u GROUP BY loadtest_id),
    f AS (SELECT r.*, m.plateau_start, m.plateau_end,
                 CASE WHEN m.plateau_start >= m.plateau_end THEN '0 no plateau (level not reached)'
                      WHEN r.start_time <  m.plateau_start THEN '1 ramp-up'
                      WHEN r.start_time <= m.plateau_end   THEN '2 plateau'
                      ELSE '3 ramp-down' END AS phase
          FROM results r JOIN m USING (loadtest_id))
    SELECT model, level, loadtest_id AS run_id, phase, count(*) AS queries,
           round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY loadtest_id), 1) AS pct_queries,
           round(greatest(any_value(plateau_end) - any_value(plateau_start), 0), 1) AS plateau_s,
           round(quantile_cont(duration, 0.5), 3) AS p50_s, round(quantile_cont(duration, 0.9), 3) AS p90_s
    FROM f GROUP BY model, level, loadtest_id, phase
    ORDER BY model, level, run_id, phase
    """, "Q17")
    if a.has_data(df):
        summary = (df.assign(in_plateau=df.pct_queries.where(df.phase.str.startswith("2"), 0.0))
                   .groupby(["model", "level", "run_id"]).agg(plateau_s=("plateau_s", "max"), pct_queries_in_plateau=("in_plateau", "sum"),
                                                              no_plateau=("phase", lambda s: bool(s.str.startswith("0").any()))).reset_index())
        summary["flag"] = np.where(summary.no_plateau | (summary.pct_queries_in_plateau < 50), "review: level poorly representative", "")
        print("Summary per run:"); display(summary)
        phase_colors = {"0 no plateau (level not reached)": GREY, "1 ramp-up": AMBER, "2 plateau": GREEN, "3 ramp-down": BLUE}
        for model, g in df.groupby("model"):
            w = g.groupby(["level", "phase"]).p90_s.mean().unstack("phase").sort_index(); cols = list(w.columns)
            fig, ax = plt.subplots(figsize=(12, 5))
            bar_grouped(ax, w.reset_index(), "level", cols, [c[2:] for c in cols], colors=[phase_colors.get(c, PALETTE[0]) for c in cols], ylabel="p90 (s)")
            ax.set_xlabel("concurrent users (nominal level)"); ax.set_title(f"Q17 · {model}: p90 per ramp phase"); finish()

## Q18 · SLA compliance per action (interaction Apdex)
**What it measures:** The share of satisfied (≤ `SLA_SEC`), tolerating (≤ 4 × SLA) and frustrated actions, and the resulting Apdex, using the serial bound and the parallel bound.

**How it is computed:** Compares `serial_time_s` (and `slowest_query_s`) of the `actions` view with `SLA_SEC`; Apdex = (satisfied + 0.5 × tolerating) / total.

**How to read it:** An interaction Apdex < 0.85 is unsatisfactory. If the parallel bound is good and the serial one bad, the problem is the serialisation of the test.

**Skew the test may introduce:** The serial bound adds queries that Power BI would launch in parallel.

**Visual:** 100 % stacked bars (green/amber/red) per level and model and an Apdex line (right axis, 0–1).

In [ ]:
with analysis("Q18", "SLA compliance per action (interaction Apdex)") as a:
    df = q(f"""
    SELECT model, level, count(*) AS actions,
           round(100.0 * avg(CASE WHEN serial_time_s <= {SLA_SEC} THEN 1 ELSE 0 END), 1) AS pct_satisfied,
           round(100.0 * avg(CASE WHEN serial_time_s > {SLA_SEC} AND serial_time_s <= 4 * {SLA_SEC} THEN 1 ELSE 0 END), 1) AS pct_tolerating,
           round(100.0 * avg(CASE WHEN serial_time_s > 4 * {SLA_SEC} THEN 1 ELSE 0 END), 1) AS pct_frustrated,
           round(avg(CASE WHEN serial_time_s <= {SLA_SEC} THEN 1.0 WHEN serial_time_s <= 4 * {SLA_SEC} THEN 0.5 ELSE 0 END), 3) AS apdex_serial,
           round(avg(CASE WHEN slowest_query_s <= {SLA_SEC} THEN 1.0 WHEN slowest_query_s <= 4 * {SLA_SEC} THEN 0.5 ELSE 0 END), 3) AS apdex_parallel_bound
    FROM actions GROUP BY model, level ORDER BY model, level
    """, "Q18")
    if a.has_data(df):
        for model, g in df.groupby("model"):
            g = g.sort_values("level"); fig, ax = plt.subplots(figsize=(12, 5.5))
            stacked_100(ax, g, [f"{l} users" for l in g.level], ["pct_satisfied", "pct_tolerating", "pct_frustrated"],
                        ["satisfied", "tolerating", "frustrated"], [GREEN, AMBER, RED])
            ax2 = ax.twinx(); ax2.grid(False); x = range(len(g))
            ax2.plot(x, g.apdex_serial, color="black", marker="o", label="Apdex (serial bound)")
            ax2.plot(x, g.apdex_parallel_bound, color="black", marker="s", ls="--", label="Apdex (parallel bound)")
            ax2.set_ylim(0, 1); ax2.set_ylabel("Apdex"); ax2.legend(loc="lower right", fontsize=8)
            ax.set_title(f"Q18 · {model}: compliance with the {SLA_SEC:g} s SLA per action"); finish()

# Block F · Test quality and regression

## Q19 · Completeness and errors
**What it measures:** Per run folder: expected users (from the `c<N>` of the name) versus CSV files written, queries per user (min/max, they must match), failed queries and lines of `error.log`. It also detects folders without any CSV.

**How it is computed:** Lists the folders with Python and joins them with a per-user consistency query over `results_all`.

**How to read it:** Expected = with results, min = max queries per user and 0 error lines. A virtual user that fails does not write its CSV: its queries do not appear in the rest of the analyses.

**Skew the test may introduce:** Folders without a CSV do not belong to any known model, so they are always shown even when filtering by `MODELS`.

**Visual:** Grouped bars expected versus with results per run (red if missing) and an error table (200 characters, UPN masked).

In [ ]:
with analysis("Q19", "Completeness and errors") as a:
    def _mask(text):
        return re.sub(r"[^\s;=]+@[^\s;]+", lambda m: "user_" + hashlib.sha1(m.group(0).encode()).hexdigest()[:6], text) if MASK_UPN else text
    rows = []
    for d in sorted(glob.glob(f"{LOGS}/{RUN_PREFIX}-*")):
        if not os.path.isdir(d): continue
        run = os.path.basename(d)
        m = re.search(r"-c(\d+)-(\d{8})-(\d{6})$", run)
        if RUN_IDS and run not in RUN_IDS: continue
        if LEVELS and m and int(m.group(1)) not in [int(l) for l in LEVELS]: continue
        if SINCE_UTC and m and datetime.strptime(m.group(2) + m.group(3), "%Y%m%d%H%M%S") < datetime.fromisoformat(SINCE_UTC.replace("Z", "+00:00")).replace(tzinfo=None): continue
        err = os.path.join(d, "error.log")
        lines = open(err, encoding="utf-8", errors="replace").read().splitlines() if os.path.exists(err) else []
        rows.append({"run_id": run, "expected_users": int(m.group(1)) if m else None,
                     "csv_users": len(glob.glob(os.path.join(d, "*_user_*.csv"))),
                     "error_lines": len([l for l in lines if l.strip()]),
                     "first_error": _mask(lines[0][:200]) if lines else ""})
    folders = pd.DataFrame(rows)
    if a.has_data(folders, "there are no run folders with that prefix"):
        cons = q("""
        SELECT loadtest_id AS run_id, model, level, count(DISTINCT thread_id) AS users_with_results,
               min(n) AS min_queries_per_user, max(n) AS max_queries_per_user
        FROM (SELECT loadtest_id, model, level, thread_id, count(*) AS n FROM results_all GROUP BY ALL)
        GROUP BY ALL
        """, show=False)
        failed = q("SELECT loadtest_id AS run_id, count(*) AS failed_queries FROM results_all WHERE status <> 'ok' GROUP BY loadtest_id", show=False)
        df = folders.merge(cons, on="run_id", how="left").merge(failed, on="run_id", how="left")
        df["failed_queries"] = df.failed_queries.fillna(0).astype(int)
        df["csv_users"] = df.csv_users.astype(int)
        df["diagnosis"] = np.select(
            [df.csv_users == 0, df.expected_users.notna() & (df.csv_users < df.expected_users),
             df.min_queries_per_user.notna() & (df.min_queries_per_user != df.max_queries_per_user), df.error_lines > 0],
            ["failed run: no CSV at all", "virtual users are missing", "users with a different number of queries", "errors in error.log"], "OK")
        RESULTS["Q19"] = df
        display(df.drop(columns=["first_error"]))
        with_error = df[df.first_error != ""][["run_id", "error_lines", "first_error"]]
        if len(with_error): print("First line of error.log (at most 200 characters, UPN masked):"); display(with_error)
        g = df.sort_values("run_id"); fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(g) + 4), 5))
        x = np.arange(len(g)); expected = g.expected_users.fillna(g.csv_users).astype(float)
        ax.bar(x - 0.2, expected, 0.4, color=GREY, label="expected users")
        ax.bar(x + 0.2, g.csv_users, 0.4, color=[RED if c < e else GREEN for c, e in zip(g.csv_users, expected)], label="CSV files with results (red = missing)")
        ax.set_xticks(x); ax.set_xticklabels([r.replace(f"{RUN_PREFIX}-", "")[-34:] for r in g.run_id], rotation=40, ha="right", fontsize=7)
        ax.set_ylabel("virtual users"); ax.legend(fontsize=8); ax.set_title("Q19 · Expected users versus users with results"); finish()

## Q20 · Run comparison (before/after regression)
**What it measures:** The variation of the p90 per visual between a reference run (`BASELINE_RUN`) and a candidate run (`CANDIDATE_RUN`), for example before and after optimising the model or changing the capacity.

**How it is computed:** p90 per visual in each run and `delta_pct` = (candidate − baseline) / baseline. With both set to `None`, it uses the two latest runs with the same model and level.

**How to read it:** Green = improvement, red = regression. Small changes in visuals with few executions are noise; look at the TOTAL row and at the visuals with many executions.

**Skew the test may introduce:** If the two runs differ in cache, capacity or think times, the comparison is not clean.

**Visual:** Horizontal diverging bars of the delta % (top `TOP_N` by |delta|) with a TOTAL row for the global p90.

In [ ]:
with analysis("Q20", "Run comparison (before/after regression)") as a:
    _base, _cand = BASELINE_RUN, CANDIDATE_RUN
    if not (_base and _cand):   # automatic: the two latest runs with the same model and level
        runs = q("SELECT loadtest_id, model, level, min(start_time) AS t0 FROM results GROUP BY ALL ORDER BY model, level, t0", show=False)
        pairs = [(g.t0.iloc[-1], g.loadtest_id.iloc[-2], g.loadtest_id.iloc[-1]) for _, g in runs.groupby(["model", "level"]) if len(g) >= 2] if len(runs) else []
        if pairs: _, _base, _cand = max(pairs)
    if not (_base and _cand):
        print("No data for Q20 · Run comparison: there are no two runs of the same model and level. Set BASELINE_RUN and CANDIDATE_RUN in Parameters.")
    else:
        print(f"Baseline:  {_base}\nCandidate: {_cand}")
        df = q(f"""
        WITH v AS (
          SELECT loadtest_id, visual_label, count(*) AS n, quantile_cont(duration, 0.9) AS p90
          FROM results WHERE loadtest_id IN ('{_base}', '{_cand}') GROUP BY ALL),
        b AS (SELECT * FROM v WHERE loadtest_id = '{_base}'),
        c AS (SELECT * FROM v WHERE loadtest_id = '{_cand}')
        SELECT coalesce(b.visual_label, c.visual_label) AS visual_label,
               round(b.p90, 3) AS p90_baseline_s, round(c.p90, 3) AS p90_candidate_s,
               round(100.0 * (c.p90 - b.p90) / nullif(b.p90, 0), 1) AS delta_pct
        FROM b FULL OUTER JOIN c ON b.visual_label = c.visual_label
        ORDER BY abs(delta_pct) DESC NULLS LAST
        """, show=False)
        if a.has_data(df, "the given runs have no successful queries"):
            tot = q(f"SELECT loadtest_id, quantile_cont(duration, 0.9) AS p90 FROM results WHERE loadtest_id IN ('{_base}', '{_cand}') GROUP BY 1", show=False).set_index("loadtest_id").p90
            if _base in tot.index and _cand in tot.index:
                row = pd.DataFrame([{"visual_label": "TOTAL (global p90)", "p90_baseline_s": round(tot[_base], 3), "p90_candidate_s": round(tot[_cand], 3),
                                     "delta_pct": round(100 * (tot[_cand] - tot[_base]) / tot[_base], 1)}])
                df = pd.concat([df.head(int(TOP_N)), row], ignore_index=True)
            RESULTS["Q20"] = df; display(df)
            fig, ax = plt.subplots(figsize=(12, max(3.5, 0.42 * len(df) + 1.8)))
            diverging_barh(ax, df, "visual_label", "delta_pct", xlabel="change in p90 (%): negative = improvement (green), positive = regression (red)")
            ax.set_title("Q20 · p90 per visual: candidate versus baseline"); finish()

## Automatic conclusions
Summarises the alerts that follow from the analyses above with the current thresholds, naming the affected model and level.

In [ ]:
# Automatic conclusions: walk through RESULTS and list alerts with the affected model and level
ALERTS = []

def alert(nn, text):
    ALERTS.append(f"[{nn}] {text}")

def _table(nn):
    df = RESULTS.get(nn)
    return df if df is not None and len(df) else None

# Q01 / Q19: users without results and errors
d = _table("Q01")
if d is not None:
    for r in d[d["users_without_results"] > 0].itertuples():
        alert("Q01", f"{r.model} · {r.level} users: {r.users_without_results} virtual users without results (run {r.run_id}).")
    for r in d[d["failed_queries"] > 0].itertuples():
        alert("Q01", f"{r.model} · {r.level} users: {r.failed_queries} failed queries (status = error) in {r.run_id}.")
d = _table("Q19")
if d is not None:
    for r in d.itertuples():
        if r.error_lines > 0 or (pd.notna(r.expected_users) and r.csv_users < r.expected_users):
            alert("Q19", f"{r.run_id}: {int(r.csv_users)} CSV files for {r.expected_users:.0f} expected users and {r.error_lines} lines in error.log.")
# Q02: first level with a knee or above the SLA
d = _table("Q02")
if d is not None:
    for model, g in d.groupby("model"):
        bad = g[g["state"] != "OK"].sort_values("level").head(1)
        for r in bad.itertuples():
            alert("Q02", f"{model}: {r.state} from {r.level} users (p90 = {r.p90_s:.2f} s, ×{r.p90_degradation_factor} over the base level).")
# Q03: saturation (throughput grows < 10 % while p90 grows > 50 %)
d = _table("Q03")
if d is not None:
    for model, g in d.sort_values("level").groupby("model"):
        a = g.reset_index(drop=True)
        for i in range(1, len(a)):
            t0, t1, p0, p1 = a.queries_per_s[i - 1], a.queries_per_s[i], a.p90_s[i - 1], a.p90_s[i]
            if t0 and p0 and t1 / t0 - 1 < 0.10 and p1 / p0 - 1 > 0.50:
                alert("Q03", f"{model}: saturation between {a.level[i - 1]} and {a.level[i]} users (throughput {100 * (t1 / t0 - 1):+.0f} %, p90 {100 * (p1 / p0 - 1):+.0f} %).")
# Q05: actions with low SLA compliance at the focus level
d = _table("Q05")
if d is not None:
    for r in at_focus(d)[lambda x: x["pct_within_sla"] < 90].itertuples():
        alert("Q05", f"{r.model} · {r.level} users: action {r.action_index} ({r.action}) meets the SLA in only {r.pct_within_sla:.0f} % of the executions.")
# Q07: visual that concentrates > 25 % of the time
d = _table("Q07")
if d is not None:
    for r in d[d["pct_total_time"] > 25].itertuples():
        alert("Q07", f"{r.model}: visual «{r.visual_label}» concentrates {r.pct_total_time:.0f} % of the total query time.")
# Q09: erratic visuals
d = _table("Q09")
if d is not None:
    n = int((d["cv"] > 1).sum())
    if n: alert("Q09", f"{n} visuals with a coefficient of variation > 1 (erratic experience); the worst: «{d.sort_values('cv', ascending=False).visual_label.iloc[0]}».")
# Q11: atypical RLS identities
d = _table("Q11")
if d is not None:
    for r in d[d["state"] == "Atypical"].itertuples():
        alert("Q11", f"{r.model} · {r.level} users: identity {r.user_key} is atypical (z = {r.z_score}, mean {r.mean_s:.2f} s).")
# Q13: unexplained overhead > 5 % of the wall clock
d = _table("Q13")
if d is not None:
    for r in d[d["unexplained_overhead_s"] > 0.05 * d["wall_s"]].itertuples():
        alert("Q13", f"{r.model} · {r.level} users: unexplained overhead of {r.unexplained_overhead_s:.1f} s ({100 * r.unexplained_overhead_s / r.wall_s:.0f} % of the wall clock).")
# Q14: think-time skew
d = _table("Q14")
if d is not None:
    for r in d.itertuples():
        if (pd.notna(r.mean_gap_offset_s) and r.mean_gap_offset_s > 0.5) or r.pauses_capped > 0:
            alert("Q14", f"{r.run_id}: mean gap offset {r.mean_gap_offset_s:.2f} s; {r.pauses_capped} pauses capped by max_think_time_sec ({r.seconds_lost_to_cap:.0f} s lost).")
# Q15: staggered-start skew
d = _table("Q15")
if d is not None:
    for run, g in d.groupby("run_id"):
        window = g["scheduled_offset_s"].max(); p95 = g["skew_vs_scheduled_s"].quantile(0.95)
        if p95 > 5 or (window > 0 and p95 > 0.10 * window):
            alert("Q15", f"{run}: start skew p95 = {p95:.1f} s (scheduled window {window:.1f} s, actual start up to {g['actual_start_s'].max():.1f} s).")
# Q16: effective concurrency far below nominal
d = _table("Q16")
if d is not None:
    for r in d[d["duty_cycle_pct"] < 10].itertuples():
        alert("Q16", f"{r.model} · {r.nominal_level} nominal users: duty cycle {r.duty_cycle_pct:.1f} % (peak of {r.peak_queries} simultaneous queries, mean in flight {r.mean_queries_in_flight}).")
# Q17: no plateau, or little load on the plateau
d = _table("Q17")
if d is not None:
    for run, g in d.groupby("run_id"):
        none = g["phase"].str.startswith("0").any()
        pct = g.loc[g["phase"].str.startswith("2"), "pct_queries"].sum()
        if none or pct < 50:
            alert("Q17", f"{run}: {'the nominal level was never reached (no plateau)' if none else f'only {pct:.0f} % of the queries happen on the plateau'}.")
# Q18: low interaction Apdex
d = _table("Q18")
if d is not None:
    for r in d[d["apdex_serial"] < 0.85].itertuples():
        alert("Q18", f"{r.model} · {r.level} users: interaction Apdex {r.apdex_serial} (parallel bound {r.apdex_parallel_bound}).")
# Q20: regressions
d = _table("Q20")
if d is not None:
    for r in d[d["delta_pct"] > 20].itertuples():
        alert("Q20", f"«{r.visual_label}» got {r.delta_pct:.0f} % worse ({r.p90_baseline_s} s → {r.p90_candidate_s} s).")
for nn, msg in ERRORS.items():
    alert(nn, f"the analysis could not be completed: {msg}")

print(f"ALERTS ({len(ALERTS)})\n" + "=" * 60 if ALERTS else "No alerts with the current thresholds.")
for a in ALERTS: print(" •", a)

## Optional persistence
With `SAVE_TO_TABLES = True` every result is saved as a Delta table `perf_qNN_<topic>` in the default lakehouse, so a Power BI report can be built on top.

In [ ]:
# Optional persistence: every result as a Delta table perf_qNN_<topic> in the default lakehouse
SLUGS = {"Q01": "executive_summary", "Q02": "degradation_curve", "Q03": "throughput_vs_latency", "Q04": "timeline",
         "Q05": "time_per_action", "Q06": "user_journey", "Q07": "slowest_visuals", "Q08": "visual_by_level",
         "Q09": "visual_stability", "Q10": "rows_vs_time", "Q11": "user_performance", "Q12": "cold_vs_warm_cache",
         "Q13": "wall_clock_breakdown", "Q14": "think_time_accuracy", "Q15": "start_skew", "Q16": "effective_concurrency",
         "Q17": "ramp_phases", "Q18": "sla_by_action", "Q19": "completeness_errors", "Q20": "run_comparison"}
TABLES_ROOT = globals().get("TABLES_ROOT")   # advanced: alternative destination path (default: the Tables of the lakehouse)

def _save(name, df):
    df = df.copy()
    for c in df.columns:                      # Delta does not accept columns of mixed objects or text NaN
        if df[c].dtype == object: df[c] = df[c].astype("string")
    if TABLES_ROOT is None and "spark" in globals():
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
        return
    from deltalake.writer import write_deltalake
    if TABLES_ROOT:
        uri, options = f"{TABLES_ROOT}/{name}", {}
    else:   # Python notebook (no Spark): deltalake over the OneLake abfss path (the mounted /lakehouse/default/Tables path does not work)
        ctx = notebookutils.runtime.context
        root = f"abfss://{ctx['currentWorkspaceId']}@onelake.dfs.fabric.microsoft.com/{ctx['defaultLakehouseId']}/Tables"
        if os.path.isdir("/lakehouse/default/Tables/dbo"): root += "/dbo"
        uri = f"{root}/{name}"
        options = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
    write_deltalake(uri, df, mode="overwrite", schema_mode="overwrite", storage_options=options or None)

if SAVE_TO_TABLES:
    saved = []
    for nn, df in RESULTS.items():
        if nn in SLUGS and len(df):
            name = f"perf_{nn.lower()}_{SLUGS[nn]}"
            _save(name, df); saved.append(name)
    print(f"Saved {len(saved)} Delta tables:", ", ".join(saved))
else:
    print("SAVE_TO_TABLES = False: nothing is saved. Set it to True in the Parameters cell to create the perf_qNN_* tables.")

## Optional JSON export
With `EXPORT_JSON = True` the results and alerts are written as `manifest.json` and `qNN_<topic>.json` under `Files/PerfScenarios/analysis/<ANALYSIS_ID>/`. The LoadTest Fabric App runs this notebook with that switch on and shows the files; you can also read them from any other tool.

In [ ]:
# Optional JSON export: manifest.json + one qNN_<topic>.json per analysis, read by the LoadTest Fabric App (UPNs already masked by q())
import json
from datetime import timezone

NOTEBOOK_VERSION = "3.6.0"
TITLES = {"Q01": "Executive summary", "Q02": "Degradation curve", "Q03": "Throughput vs latency", "Q04": "Timeline",
          "Q05": "Time per action", "Q06": "User journey", "Q07": "Slowest visuals", "Q08": "Visual by level",
          "Q09": "Visual stability", "Q10": "Rows vs time", "Q11": "Performance per user", "Q12": "Cold vs warm cache",
          "Q13": "Wall-clock breakdown", "Q14": "Think-time accuracy", "Q15": "Start skew", "Q16": "Effective concurrency",
          "Q17": "Ramp phases", "Q18": "SLA by action", "Q19": "Completeness and errors", "Q20": "Run comparison"}
BLOCKS_OF = {**{f"Q0{n}": "A" for n in (1, 2, 3, 4)}, **{f"Q0{n}": "B" for n in (5, 6)}, **{f"Q0{n}": "C" for n in (7, 8, 9)}, "Q10": "C",
             "Q12": "C", "Q11": "D", **{f"Q{n}": "E" for n in (13, 14, 15, 16, 17)}, **{f"Q{n}": "F" for n in (18, 19, 20)}}
# How the app draws each result from its table: the chart is data only, no code
CHARTS = {
    "Q01": {"type": "grouped_bar", "group_by": "model", "x": "level", "y": ["p50_s", "p90_s", "p95_s"], "hline": "APDEX_T"},
    "Q02": {"type": "line", "x": "level", "y": ["p50_s", "p90_s", "p95_s"], "series": "model", "hline": "SLA_SEC"},
    "Q03": {"type": "combo", "group_by": "model", "x": "level", "bars": ["queries_per_s"], "lines": ["p90_s"]},
    "Q04": {"type": "small_multiples", "group_by": "run_id", "x": "sec_from_start", "y": ["active_users", "queries_in_flight", "p90_s"]},
    "Q05": {"type": "hbar", "group_by": "model", "focus": True, "label": "action", "value": "p90_serial_s", "marker": "p90_parallel_bound_s", "vline": "SLA_SEC"},
    "Q06": {"type": "grouped_bar", "group_by": "model", "x": "level", "y": ["p50_query_time_s", "p90_query_time_s"]},
    "Q07": {"type": "hbar", "group_by": "model", "label": "visual_label", "value": "p90_s", "color": "pct_total_time"},
    "Q08": {"type": "heatmap", "group_by": "model", "rows": "visual_label", "cols": "level", "value": "p90_s"},
    "Q09": {"type": "scatter", "group_by": "model", "x": "mean_s", "y": "cv", "size": "executions", "label": "visual_label"},
    "Q10": {"type": "scatter", "group_by": "model", "x": "mean_rows", "y": "p50_s", "size": "executions", "label": "visual_label", "logx": True, "logy": True},
    "Q11": {"type": "hbar", "group_by": "model", "label": "user_key", "value": "p90_s", "flag": {"column": "state", "value": "Atypical"}},
    "Q12": {"type": "grouped_bar", "group_by": "model", "x": "level", "series": "cache_state", "y": ["p90_s"], "agg": "mean"},
    "Q13": {"type": "stacked100", "group_by": None, "x": ["model", "level"], "y": ["initial_offset_s", "queries_s", "think_time_s", "delay_waits_s", "unexplained_overhead_s"]},
    "Q14": {"type": "grouped_bar", "x": "run_id", "y": ["mean_sleep_error_ms", "max_sleep_error_ms"]},
    "Q15": {"type": "small_multiples", "group_by": "run_id", "kind": "scatter", "x": "thread_id", "y": ["actual_start_s", "scheduled_offset_s"]},
    "Q16": {"type": "combo", "group_by": "model", "x": "nominal_level", "bars": ["nominal_level", "peak_users", "peak_queries", "mean_queries_in_flight"], "lines": ["duty_cycle_pct"]},
    "Q17": {"type": "grouped_bar", "group_by": "model", "x": "level", "series": "phase", "y": ["p90_s"], "agg": "mean"},
    "Q18": {"type": "stacked100", "group_by": "model", "x": ["level"], "y": ["pct_satisfied", "pct_tolerating", "pct_frustrated"], "lines": ["apdex_serial", "apdex_parallel_bound"]},
    "Q19": {"type": "grouped_bar", "x": "run_id", "y": ["expected_users", "csv_users"]},
    "Q20": {"type": "diverging_bar", "label": "visual_label", "value": "delta_pct"},
}
PARAMETER_NAMES = ["RUN_PREFIX", "SINCE_UTC", "RUN_IDS", "MODELS", "LEVELS", "FOCUS_LEVEL", "SLA_SEC", "APDEX_T", "KNEE_FACTOR", "BUCKET_SEC",
                   "TOP_N", "TIMELINE_MAX_RUNS", "BASELINE_RUN", "CANDIDATE_RUN", "MASK_UPN"]

def _alert_record(text):
    query = re.match(r"\[(Q\d\d)\]\s*(.*)", text, re.S)
    nn, body = (query.group(1), query.group(2)) if query else (None, text)
    where = re.match(r"(.+?) · (\d+) (?:nominal )?users", body)
    model_only = None if where else re.match(r"([^:·]+?): ", body)
    return {"query": nn, "model": where.group(1) if where else (model_only.group(1) if model_only else None),
            "level": int(where.group(2)) if where else None,
            "severity": "high" if nn in ("Q01", "Q19") else "warning", "text": body}

def _records(df):
    return json.loads(df.to_json(orient="records", date_format="iso"))

def _column_types(df):
    kind = {"i": "integer", "u": "integer", "f": "number", "b": "boolean", "M": "datetime"}
    return [{"name": c, "type": kind.get(df[c].dtype.kind, "string")} for c in df.columns]

if EXPORT_JSON:
    stamp = datetime.now(timezone.utc)
    analysis_id = ANALYSIS_ID or f"{RUN_PREFIX}-{stamp.strftime('%Y%m%d-%H%M%S')}"
    out_dir = os.path.join(EXPORT_ROOT, analysis_id)
    os.makedirs(out_dir, exist_ok=True)
    queries = []
    for number in range(1, 21):
        nn = f"Q{number:02d}"
        df = RESULTS.get(nn)
        entry = {"id": nn, "block": BLOCKS_OF[nn], "title": TITLES[nn], "chart": CHARTS[nn], "file": None, "rows": 0, "columns": []}
        if nn in ERRORS:
            entry.update(status="error", message=ERRORS[nn])
        elif df is None or len(df) == 0:
            entry.update(status="no_data", message="The query returned no rows with the current filters.")
        else:
            file_name = f"{nn.lower()}_{SLUGS[nn]}.json"
            with open(os.path.join(out_dir, file_name), "w", encoding="utf-8") as handle:
                json.dump(_records(df), handle, ensure_ascii=False)
            entry.update(status="ok", message="", file=file_name, rows=int(len(df)), columns=_column_types(df))
        queries.append(entry)
    manifest = {"notebook": "AnalyzeReportPerformance_v3", "notebook_version": NOTEBOOK_VERSION, "analysis_id": analysis_id,
                "generated_utc": stamp.isoformat(), "parameters": {k: globals().get(k) for k in PARAMETER_NAMES},
                "focus": FOCUS, "runs": sorted(set(RESULTS["Q01"]["run_id"])) if "Q01" in RESULTS else [],
                "alerts": [_alert_record(a) for a in ALERTS], "queries": queries}
    with open(os.path.join(out_dir, "manifest.json"), "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, default=str)
    print(f"Exported {sum(q['status'] == 'ok' for q in queries)} results and {len(ALERTS)} alerts to {out_dir}")
else:
    print("EXPORT_JSON = False: no JSON is written. The LoadTest Fabric App sets it to True when it runs this notebook.")